In [12]:
import os
import json
import hashlib
import secrets
import subprocess
import platform
import tkinter as tk
from tkinter import ttk, filedialog, messagebox

from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes


G = {
    "root": None,
    "frames": {},
    "status": None,

    # 加密頁面狀態
    "secret_path": None,
    "output_folder": None,
    "password": None,
    "container_mb": None,
    "device_binding_mode": None,
    "recovery_mode": None,

    # 解密頁面狀態
    "extract_container_path": None,
    "extract_metadata_path": None,
    "extract_output_folder": None,
    "extract_password": None,
    "decrypt_mode": None,
    "recovery_key_input": None,
}


# ============================================================
# UI 基礎控制與主題設定
# ============================================================

def setup_theme(root):
    """設定現代化乾淨主題"""
    style = ttk.Style(root)
    # 使用 clam 主題作為自訂基礎，因為它跨平台且去除了老舊的 3D 邊框
    style.theme_use("clam")

    # 定義色系
    BG_COLOR = "#F3F4F6"      # 淺灰背景
    CARD_BG = "#FFFFFF"       # 純白卡片背景
    TEXT_MAIN = "#1F2937"     # 深灰主文字
    TEXT_MUTED = "#6B7280"    # 淺灰輔助文字
    ACCENT_COLOR = "#2563EB"  # 現代藍主色
    ACCENT_HOVER = "#1D4ED8"  # 按鈕懸停色

    root.configure(bg=BG_COLOR)

    # 基礎設定
    style.configure(".", background=BG_COLOR, foreground=TEXT_MAIN, font=("Microsoft JhengHei UI", 11))
    style.configure("TFrame", background=BG_COLOR)
    
    # 卡片風格 Frame (用於包裝表單)
    style.configure("Card.TFrame", background=CARD_BG)

    # 標籤風格
    style.configure("TLabel", background=BG_COLOR, foreground=TEXT_MAIN)
    style.configure("Card.TLabel", background=CARD_BG, foreground=TEXT_MAIN)
    style.configure("Title.TLabel", font=("Microsoft JhengHei UI", 24, "bold"), foreground="#111827")
    style.configure("Subtitle.TLabel", font=("Microsoft JhengHei UI", 12), foreground=TEXT_MUTED)
    style.configure("Status.TLabel", background="#E5E7EB", foreground=TEXT_MAIN, padding=5, font=("Microsoft JhengHei UI", 10))

    # 一般按鈕風格
    style.configure("TButton", font=("Microsoft JhengHei UI", 10), padding=6, background="#FFFFFF", bordercolor="#D1D5DB")
    style.map("TButton", background=[("active", "#F3F4F6")])

    # 主按鈕風格 (Accent)
    style.configure("Accent.TButton", font=("Microsoft JhengHei UI", 11, "bold"), foreground="white", background=ACCENT_COLOR)
    style.map("Accent.TButton", background=[("active", ACCENT_HOVER)])

    # 輸入框與下拉選單風格
    style.configure("TEntry", padding=5)
    style.configure("TCombobox", padding=5)


def set_status(msg):
    if G["status"] is not None:
        G["status"].set(f"  狀態：{msg}")


def show(name):
    G["frames"][name].tkraise()
    set_status(f"目前頁面 - {name}")


def build_menu():
    mbar = tk.Menu(G["root"])

    m = tk.Menu(mbar, tearoff=0)
    m.add_command(label="首頁", command=lambda: show("首頁"))
    m.add_command(label="加密藏入", command=lambda: show("加密藏入"))
    m.add_command(label="解密取出", command=lambda: show("解密取出"))
    m.add_separator()
    m.add_command(label="安全設定說明", command=lambda: show("安全設定說明"))
    m.add_command(label="系統說明", command=lambda: show("系統說明"))
    m.add_separator()
    m.add_command(label="離開", command=G["root"].quit)

    mbar.add_cascade(label="功能選單", menu=m)
    G["root"].config(menu=mbar)


# ============================================================
# 基礎工具函式
# ============================================================

def yes_no_value(text):
    return text in ["開啟", "產生", "啟用", "是"]


def safe_filename(name):
    bad = '<>:"/\\|?*'
    out = ""
    for ch in name:
        if ch in bad:
            out += "_"
        else:
            out += ch
    return out


def run_command(cmd, timeout=8):
    try:
        creationflags = 0
        if os.name == "nt":
            creationflags = subprocess.CREATE_NO_WINDOW
        result = subprocess.run(
            cmd, capture_output=True, text=True, errors="replace",
            timeout=timeout, creationflags=creationflags
        )
        text = ""
        if result.stdout: text += result.stdout
        if result.stderr: text += result.stderr
        return text.strip()
    except Exception:
        return ""


def sha256_bytes(data):
    return hashlib.sha256(data).digest()


def sha256_hex(data):
    return hashlib.sha256(data).hexdigest()


# ============================================================
# Device Fingerprint
# ============================================================

def read_windows_machine_guid():
    if os.name != "nt": return ""
    try:
        import winreg
        key = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r"SOFTWARE\Microsoft\Cryptography")
        value, _ = winreg.QueryValueEx(key, "MachineGuid")
        winreg.CloseKey(key)
        return str(value).strip()
    except Exception:
        return ""

def powershell_value(command):
    if os.name != "nt": return ""
    full_cmd = ["powershell", "-NoProfile", "-ExecutionPolicy", "Bypass", "-Command", command]
    output = run_command(full_cmd)
    lines = [line.strip() for line in output.splitlines() if line.strip()]
    if not lines: return ""
    return lines[0]

def get_bios_uuid(): return powershell_value("(Get-CimInstance Win32_ComputerSystemProduct).UUID")
def get_bios_serial(): return powershell_value("(Get-CimInstance Win32_BIOS).SerialNumber")
def get_baseboard_serial(): return powershell_value("(Get-CimInstance Win32_BaseBoard).SerialNumber")

def get_device_fingerprint_raw():
    machine_guid = read_windows_machine_guid()
    bios_uuid = get_bios_uuid()
    bios_serial = get_bios_serial()
    board_serial = get_baseboard_serial()
    parts = {
        "os": platform.system(),
        "machine_guid": machine_guid,
        "bios_uuid": bios_uuid,
        "bios_serial": bios_serial,
        "baseboard_serial": board_serial,
    }
    raw = json.dumps(parts, sort_keys=True, ensure_ascii=False)
    return raw, parts

def get_device_hash():
    raw, parts = get_device_fingerprint_raw()
    return sha256_bytes(raw.encode("utf-8")), parts


# ============================================================
# 密碼學核心
# ============================================================

def bytes_to_bits(data):
    bits = []
    for byte in data:
        for i in range(8):
            bits.append((byte >> i) & 1)
    return bits

def bits_to_bytes(bits):
    out = bytearray()
    for i in range(0, len(bits), 8):
        value = 0
        chunk = bits[i:i + 8]
        for bit_index, bit in enumerate(chunk):
            value |= (bit << bit_index)
        out.append(value)
    return bytes(out)

def set_bit(byte_value, bit_index, secret_bit):
    if secret_bit == 1:
        return byte_value | (1 << bit_index)
    else:
        return byte_value & ~(1 << bit_index)

def get_bit(byte_value, bit_index):
    return (byte_value >> bit_index) & 1

def shake_stream(seed):
    counter = 0
    while True:
        data = seed + counter.to_bytes(8, "big")
        block = hashlib.shake_256(data).digest(64)
        for b in block: yield b
        counter += 1

def random_int_from_stream(stream, max_value):
    raw = bytearray()
    for _ in range(8): raw.append(next(stream))
    value = int.from_bytes(raw, "big")
    return value % max_value

def generate_positions(seed, total_bytes, bit_count):
    capacity = total_bytes * 8
    if bit_count > capacity:
        raise ValueError(f"容器容量不夠。\n\n需要 bit 數：{bit_count}\n容器容量 bit 數：{capacity}\n\n請增加 container size。")
    stream = shake_stream(seed)
    used = set()
    positions = []
    while len(positions) < bit_count:
        global_bit_index = random_int_from_stream(stream, capacity)
        if global_bit_index in used: continue
        used.add(global_bit_index)
        byte_index = global_bit_index // 8
        bit_index = global_bit_index % 8
        positions.append((byte_index, bit_index))
    return positions

def split_content_key_material(key_material):
    if len(key_material) != 96:
        raise ValueError("key material 長度錯誤，必須是 96 bytes")
    return key_material[0:32], key_material[32:64], key_material[64:96]

def derive_access_key(password, salt, access_secret):
    if isinstance(access_secret, str):
        access_secret = access_secret.encode("utf-8")
    source = password.encode("utf-8") + b"\x00PHANTASM-V2\x00" + access_secret
    return hashlib.pbkdf2_hmac("sha256", source, salt, 300_000, dklen=32)

def aes_gcm_encrypt(key, data, aad=None):
    nonce = get_random_bytes(12)
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    if aad is not None: cipher.update(aad)
    ciphertext, tag = cipher.encrypt_and_digest(data)
    return nonce, ciphertext, tag

def aes_gcm_decrypt(key, nonce, ciphertext, tag, aad=None):
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    if aad is not None: cipher.update(aad)
    return cipher.decrypt_and_verify(ciphertext, tag)

def wrap_key_material(key_material, password, access_secret, wrap_type):
    salt = get_random_bytes(16)
    access_key = derive_access_key(password, salt, access_secret)
    aad = f"PHANTASM-V2-WRAP-{wrap_type}".encode("utf-8")
    nonce, wrapped, tag = aes_gcm_encrypt(access_key, key_material, aad=aad)
    return {
        "type": wrap_type, "salt_hex": salt.hex(),
        "nonce_hex": nonce.hex(), "tag_hex": tag.hex(),
        "wrapped_key_material_hex": wrapped.hex()
    }

def unwrap_key_material(wrap_info, password, access_secret):
    wrap_type = wrap_info["type"]
    salt = bytes.fromhex(wrap_info["salt_hex"])
    nonce = bytes.fromhex(wrap_info["nonce_hex"])
    tag = bytes.fromhex(wrap_info["tag_hex"])
    wrapped = bytes.fromhex(wrap_info["wrapped_key_material_hex"])
    access_key = derive_access_key(password, salt, access_secret)
    aad = f"PHANTASM-V2-WRAP-{wrap_type}".encode("utf-8")
    return aes_gcm_decrypt(access_key, nonce, wrapped, tag, aad=aad)

def generate_recovery_key():
    raw = secrets.token_hex(24).upper()
    groups = [raw[i:i + 4] for i in range(0, len(raw), 4)]
    return "PHANTASM-RECOVERY-" + "-".join(groups)

def save_recovery_key_file(output_folder, recovery_key):
    path = os.path.join(output_folder, "phantasm_recovery_key.txt")
    content = f"""Project Phantasm v2.2 Recovery Key

請妥善保存這組 Recovery Key。

如果原電腦故障、Windows 重灌、主機板更換，
或無法取得原本的裝置指紋，
可以使用此 Recovery Key 搭配原本密碼解密。

注意：
1. Recovery Key 遺失後無法復原。
2. 請不要和 container.bin、metadata.json 放在同一個不安全位置。
3. 建議離線保存。

Recovery Key:
{recovery_key}
"""
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    return path


# ============================================================
# 加密藏入核心
# ============================================================

def encrypt_and_embed_v2(secret_path, password, container_mb, output_folder, device_binding, recovery_enabled):
    with open(secret_path, "rb") as f: secret_data = f.read()

    content_key_material = get_random_bytes(96)
    aes_key, position_seed, meta_key = split_content_key_material(content_key_material)

    content_nonce, ciphertext, content_tag = aes_gcm_encrypt(aes_key, secret_data, aad=b"PHANTASM-V2-CONTENT")

    container_size = int(container_mb * 1024 * 1024)
    container = bytearray(os.urandom(container_size))

    cipher_bits = bytes_to_bits(ciphertext)
    positions = generate_positions(seed=position_seed, total_bytes=len(container), bit_count=len(cipher_bits))

    for secret_bit, (byte_index, bit_index) in zip(cipher_bits, positions):
        container[byte_index] = set_bit(container[byte_index], bit_index, secret_bit)

    wrapped_keys = []
    device_parts = {}
    device_check_salt = get_random_bytes(16)
    device_check_hex = ""

    if device_binding:
        device_hash, device_parts = get_device_hash()
        primary_access_secret = device_hash
        device_check_hex = sha256_hex(device_hash + device_check_salt)
        primary_wrap_type = "primary_device_bound"
    else:
        primary_access_secret = b"NO_DEVICE_BINDING_V1"
        primary_wrap_type = "primary_password_only"

    primary_wrap = wrap_key_material(content_key_material, password, primary_access_secret, primary_wrap_type)
    wrapped_keys.append(primary_wrap)

    recovery_key = None
    recovery_key_path = None
    recovery_check_salt = None
    recovery_check_hex = ""

    if recovery_enabled:
        recovery_key = generate_recovery_key()
        recovery_secret = recovery_key.encode("utf-8")
        recovery_wrap = wrap_key_material(content_key_material, password, recovery_secret, "recovery_key")
        wrapped_keys.append(recovery_wrap)
        recovery_check_salt = get_random_bytes(16)
        recovery_check_hex = sha256_hex(recovery_secret + recovery_check_salt)
        recovery_key_path = save_recovery_key_file(output_folder, recovery_key)

    metadata = {
        "version": "Phantasm-v2.2-No-BitLocker-Device-Bound-Random-Container",
        "created_by": "Project Phantasm v2.2",
        "ciphertext_length": len(ciphertext),
        "container_size": len(container),
        "original_filename": os.path.basename(secret_path),
        "content_nonce_hex": content_nonce.hex(),
        "content_tag_hex": content_tag.hex(),
        "kdf": "PBKDF2-HMAC-SHA256", "iterations": 300000,
        "cipher": "AES-256-GCM", "position_generator": "SHAKE256",
        "coordinate_type": "(byte_index, bit_index)",
        "device_binding": device_binding,
        "device_fingerprint_method": "MachineGuid+BIOS_UUID+BIOS_Serial+BoardSerial",
        "device_check_salt_hex": device_check_salt.hex(),
        "device_hash_check_hex": device_check_hex,
        "recovery_enabled": recovery_enabled,
        "recovery_check_salt_hex": recovery_check_salt.hex() if recovery_check_salt else "",
        "recovery_key_check_hex": recovery_check_hex,
        "recovery_hint": "Recovery Key was generated at encryption time." if recovery_enabled else "",
        "wrapped_keys": wrapped_keys,
        "security_note": {
            "aes_gcm": "保護內容機密性與完整性",
            "scatter": "密文 bit 被分散藏入 random container",
            "device_binding": "啟用時需要原電腦裝置指紋",
            "recovery_key": "可在原電腦失效時恢復解密能力",
            "bitlocker": "此版本已移除 BitLocker 檢查功能"
        }
    }

    container_path = os.path.join(output_folder, "phantasm_container.bin")
    metadata_path = os.path.join(output_folder, "phantasm_metadata.json")
    with open(container_path, "wb") as f: f.write(container)
    with open(metadata_path, "w", encoding="utf-8") as f: json.dump(metadata, f, ensure_ascii=False, indent=4)

    return {"container_path": container_path, "metadata_path": metadata_path, "recovery_key_path": recovery_key_path, "recovery_key": recovery_key}


# ============================================================
# 解密取出核心
# ============================================================

def find_wrap(metadata, wrap_type):
    for item in metadata.get("wrapped_keys", []):
        if item.get("type") == wrap_type: return item
    return None

def try_unwrap_primary(metadata, password):
    device_binding = metadata.get("device_binding", False)
    if device_binding:
        wrap = find_wrap(metadata, "primary_device_bound")
        if wrap is None: raise RuntimeError("metadata 中找不到裝置綁定 key wrap。")
        device_hash, _ = get_device_hash()
        check_salt_hex = metadata.get("device_check_salt_hex", "")
        check_hex = metadata.get("device_hash_check_hex", "")
        if check_salt_hex and check_hex:
            check_salt = bytes.fromhex(check_salt_hex)
            if sha256_hex(device_hash + check_salt) != check_hex:
                raise RuntimeError("目前電腦的裝置指紋與加密時不一致。\n如果你已更換電腦或重灌系統，請使用 Recovery Key 解密。")
        return unwrap_key_material(wrap, password, device_hash)
    else:
        wrap = find_wrap(metadata, "primary_password_only")
        if wrap is None: raise RuntimeError("metadata 中找不到 password-only key wrap。")
        return unwrap_key_material(wrap, password, b"NO_DEVICE_BINDING_V1")

def try_unwrap_recovery(metadata, password, recovery_key):
    if not metadata.get("recovery_enabled", False): raise RuntimeError("此檔案未啟用 Recovery Key。")
    if not recovery_key: raise RuntimeError("請輸入 Recovery Key。")
    wrap = find_wrap(metadata, "recovery_key")
    if wrap is None: raise RuntimeError("metadata 中找不到 Recovery Key wrap。")
    recovery_secret = recovery_key.strip().encode("utf-8")
    check_salt_hex = metadata.get("recovery_check_salt_hex", "")
    check_hex = metadata.get("recovery_key_check_hex", "")
    if check_salt_hex and check_hex:
        check_salt = bytes.fromhex(check_salt_hex)
        if sha256_hex(recovery_secret + check_salt) != check_hex:
            raise RuntimeError("Recovery Key 不正確。")
    return unwrap_key_material(wrap, password, recovery_secret)

def extract_and_decrypt_v2(container_path, metadata_path, password, output_folder, decrypt_mode, recovery_key):
    with open(container_path, "rb") as f: container = bytearray(f.read())
    with open(metadata_path, "r", encoding="utf-8") as f: metadata = json.load(f)

    if not metadata.get("version", "").startswith("Phantasm-v2"):
        raise RuntimeError("metadata 版本不是 Project Phantasm v2 系列。")

    key_material = None
    errors = []

    if decrypt_mode == "正常解密":
        key_material = try_unwrap_primary(metadata, password)
    elif decrypt_mode == "Recovery Key 解密":
        key_material = try_unwrap_recovery(metadata, password, recovery_key)
    else:
        try:
            key_material = try_unwrap_primary(metadata, password)
        except Exception as e:
            errors.append("正常解密失敗：" + str(e))
            if recovery_key.strip():
                try:
                    key_material = try_unwrap_recovery(metadata, password, recovery_key)
                except Exception as e2:
                    errors.append("Recovery 解密失敗：" + str(e2))

    if key_material is None: raise RuntimeError("\n\n".join(errors))

    aes_key, position_seed, meta_key = split_content_key_material(key_material)
    ciphertext_length = int(metadata["ciphertext_length"])
    positions = generate_positions(seed=position_seed, total_bytes=len(container), bit_count=ciphertext_length * 8)
    
    extracted_bits = [get_bit(container[byte_index], bit_index) for byte_index, bit_index in positions]
    ciphertext = bits_to_bytes(extracted_bits)

    content_nonce = bytes.fromhex(metadata["content_nonce_hex"])
    content_tag = bytes.fromhex(metadata["content_tag_hex"])
    plaintext = aes_gcm_decrypt(aes_key, content_nonce, ciphertext, content_tag, aad=b"PHANTASM-V2-CONTENT")

    original_filename = safe_filename(metadata.get("original_filename", "recovered_file.bin"))
    output_path = os.path.join(output_folder, "recovered_" + original_filename)
    with open(output_path, "wb") as f: f.write(plaintext)

    return output_path


# ============================================================
# 檔案選擇
# ============================================================

def select_secret_file(label):
    path = filedialog.askopenfilename(title="選擇要加密的檔案")
    if path:
        G["secret_path"] = path
        label.config(text=os.path.basename(path)) # 為了版面乾淨只顯示檔名
        set_status("已選擇秘密檔案")

def select_output_folder(label):
    path = filedialog.askdirectory(title="選擇輸出資料夾")
    if path:
        G["output_folder"] = path
        label.config(text=path)
        set_status("已選擇輸出資料夾")

def select_extract_container(label):
    path = filedialog.askopenfilename(title="選擇 container.bin", filetypes=[("BIN files", "*.bin"), ("All files", "*.*")])
    if path:
        G["extract_container_path"] = path
        label.config(text=os.path.basename(path))
        set_status("已選擇 container 檔案")

def select_extract_metadata(label):
    path = filedialog.askopenfilename(title="選擇 metadata.json", filetypes=[("JSON files", "*.json"), ("All files", "*.*")])
    if path:
        G["extract_metadata_path"] = path
        label.config(text=os.path.basename(path))
        set_status("已選擇 metadata 檔案")

def select_extract_output_folder(label):
    path = filedialog.askdirectory(title="選擇還原輸出資料夾")
    if path:
        G["extract_output_folder"] = path
        label.config(text=path)
        set_status("已選擇還原輸出資料夾")


# ============================================================
# UI 動作
# ============================================================

def run_encrypt():
    if not G["secret_path"]: return messagebox.showerror("錯誤", "請選擇要加密的檔案")
    if not G["output_folder"]: return messagebox.showerror("錯誤", "請選擇輸出資料夾")
    password = G["password"].get()
    if not password: return messagebox.showerror("錯誤", "請輸入密碼")
    
    try:
        container_mb = float(G["container_mb"].get())
        if container_mb <= 0: raise ValueError
    except Exception:
        return messagebox.showerror("錯誤", "容器大小必須是正數 MB")

    device_binding = yes_no_value(G["device_binding_mode"].get())
    recovery_enabled = yes_no_value(G["recovery_mode"].get())

    try:
        set_status("正在加密並藏入資料...")
        result = encrypt_and_embed_v2(G["secret_path"], password, container_mb, G["output_folder"], device_binding, recovery_enabled)
        msg = f"加密藏入完成！\n\n容器檔：{result['container_path']}\nmetadata：{result['metadata_path']}\n"
        if result["recovery_key_path"]:
            msg += f"\nRecovery Key 檔案：{result['recovery_key_path']}\n\n請務必妥善保存 Recovery Key。"
        messagebox.showinfo("完成", msg)
        set_status("加密藏入完成")
    except Exception as e:
        messagebox.showerror("加密失敗", str(e))
        set_status("加密藏入失敗")

def run_extract():
    if not G["extract_container_path"]: return messagebox.showerror("錯誤", "請選擇 container.bin")
    if not G["extract_metadata_path"]: return messagebox.showerror("錯誤", "請選擇 metadata.json")
    if not G["extract_output_folder"]: return messagebox.showerror("錯誤", "請選擇輸出資料夾")
    password = G["extract_password"].get()
    if not password: return messagebox.showerror("錯誤", "請輸入密碼")

    decrypt_mode = G["decrypt_mode"].get()
    recovery_key = G["recovery_key_input"].get().strip()

    if decrypt_mode == "Recovery Key 解密" and not recovery_key:
        return messagebox.showerror("錯誤", "此模式需要輸入 Recovery Key")

    try:
        set_status("正在取出並解密資料...")
        output_path = extract_and_decrypt_v2(
            G["extract_container_path"], G["extract_metadata_path"], password,
            G["extract_output_folder"], decrypt_mode, recovery_key
        )
        messagebox.showinfo("完成", f"解密完成！\n\n還原檔案：{output_path}")
        set_status("解密取出完成")
    except Exception as e:
        messagebox.showerror("解密失敗", f"可能原因：\n1. 密碼錯誤\n2. 不是原本的電腦\n3. Recovery Key 錯誤\n4. 檔案被修改\n\n詳細：{str(e)}")
        set_status("解密取出失敗")

def show_device_fingerprint():
    try:
        device_hash, parts = get_device_hash()
        text = f"目前裝置指紋摘要：\n{device_hash.hex()[:32]}...\n\n使用的裝置資訊：\n"
        for k, v in parts.items(): text += f"- {k}: {v}\n"
        messagebox.showinfo("裝置指紋 (Device Fingerprint)", text)
    except Exception as e:
        messagebox.showerror("錯誤", str(e))


# ============================================================
# UI 分頁 - 美化版
# ============================================================

def frame_home(parent):
    f = ttk.Frame(parent)
    
    # 建立內容置中容器
    center = ttk.Frame(f)
    center.place(relx=0.5, rely=0.45, anchor="center")

    ttk.Label(center, text="Project Phantasm v2.2", style="Title.TLabel").pack(pady=(0, 5))
    ttk.Label(center, text="Device-Bound Random Container Encryption System", style="Subtitle.TLabel").pack(pady=(0, 25))

    card = ttk.Frame(center, style="Card.TFrame", padding=30)
    card.pack(fill="x", pady=10)

    ttk.Label(card, text="本系統提供 AES-GCM 高強度加密，搭配裝置綁定與密文隨機藏入機制。", style="Card.TLabel").pack(pady=(0,20))

    btn_frame = ttk.Frame(card, style="Card.TFrame")
    btn_frame.pack(fill="x")
    
    ttk.Button(btn_frame, text="開始加密藏入", style="Accent.TButton", width=18, command=lambda: show("加密藏入")).pack(side="left", padx=10)
    ttk.Button(btn_frame, text="解密取出檔案", width=18, command=lambda: show("解密取出")).pack(side="right", padx=10)

    ttk.Label(center, text="此版本已移除 BitLocker 檢查功能", foreground="#9CA3AF").pack(pady=20)

    return f


def frame_encrypt(parent):
    f = ttk.Frame(parent, padding=30)
    
    ttk.Label(f, text="加密藏入", style="Title.TLabel").pack(anchor="w", pady=(0,5))
    ttk.Label(f, text="將機密檔案轉換為隨機容器，並綁定當前裝置。", style="Subtitle.TLabel").pack(anchor="w", pady=(0,20))

    # 第一段：檔案選擇卡片
    card1 = ttk.Frame(f, style="Card.TFrame", padding=20)
    card1.pack(fill="x", pady=10)

    row1 = ttk.Frame(card1, style="Card.TFrame")
    row1.pack(fill="x", pady=5)
    ttk.Button(row1, text="選擇要加密的檔案", width=20, command=lambda: select_secret_file(file_lbl)).pack(side="left", padx=(0,15))
    file_lbl = ttk.Label(row1, text="尚未選擇", style="Card.TLabel", foreground="#6B7280")
    file_lbl.pack(side="left")

    row2 = ttk.Frame(card1, style="Card.TFrame")
    row2.pack(fill="x", pady=5)
    ttk.Button(row2, text="選擇輸出資料夾", width=20, command=lambda: select_output_folder(folder_lbl)).pack(side="left", padx=(0,15))
    folder_lbl = ttk.Label(row2, text="尚未選擇", style="Card.TLabel", foreground="#6B7280")
    folder_lbl.pack(side="left")

    # 第二段：參數設定卡片
    card2 = ttk.Frame(f, style="Card.TFrame", padding=20)
    card2.pack(fill="x", pady=10)

    grid = ttk.Frame(card2, style="Card.TFrame")
    grid.pack(fill="x")
    grid.columnconfigure(1, weight=1)

    ttk.Label(grid, text="加密密碼", style="Card.TLabel").grid(row=0, column=0, pady=10, sticky="e", padx=(0,15))
    G["password"] = tk.StringVar()
    ttk.Entry(grid, textvariable=G["password"], show="●", font=("Arial", 11)).grid(row=0, column=1, sticky="we", pady=10)

    ttk.Label(grid, text="容器大小 (MB)", style="Card.TLabel").grid(row=1, column=0, pady=10, sticky="e", padx=(0,15))
    G["container_mb"] = tk.StringVar(value="10")
    ttk.Entry(grid, textvariable=G["container_mb"]).grid(row=1, column=1, sticky="we", pady=10)

    ttk.Label(grid, text="裝置綁定模式", style="Card.TLabel").grid(row=2, column=0, pady=10, sticky="e", padx=(0,15))
    G["device_binding_mode"] = tk.StringVar(value="開啟")
    ttk.Combobox(grid, textvariable=G["device_binding_mode"], values=["開啟", "關閉"], state="readonly").grid(row=2, column=1, sticky="we", pady=10)

    ttk.Label(grid, text="備援金鑰 (Recovery)", style="Card.TLabel").grid(row=3, column=0, pady=10, sticky="e", padx=(0,15))
    G["recovery_mode"] = tk.StringVar(value="產生")
    ttk.Combobox(grid, textvariable=G["recovery_mode"], values=["產生", "不產生"], state="readonly").grid(row=3, column=1, sticky="we", pady=10)

    # 底部按鈕
    bottom = ttk.Frame(f)
    bottom.pack(fill="x", pady=20)
    ttk.Button(bottom, text="返回首頁", command=lambda: show("首頁")).pack(side="left")
    ttk.Button(bottom, text="查看裝置指紋", command=show_device_fingerprint).pack(side="right", padx=(10,0))
    ttk.Button(bottom, text="執行加密藏入", style="Accent.TButton", command=run_encrypt).pack(side="right")

    return f


def frame_extract(parent):
    f = ttk.Frame(parent, padding=30)
    
    ttk.Label(f, text="解密取出", style="Title.TLabel").pack(anchor="w", pady=(0,5))
    ttk.Label(f, text="讀取隨機容器與 Metadata 以還原原始檔案。", style="Subtitle.TLabel").pack(anchor="w", pady=(0,20))

    # 檔案選擇卡片
    card1 = ttk.Frame(f, style="Card.TFrame", padding=20)
    card1.pack(fill="x", pady=10)

    row1 = ttk.Frame(card1, style="Card.TFrame")
    row1.pack(fill="x", pady=5)
    ttk.Button(row1, text="選擇 Container.bin", width=20, command=lambda: select_extract_container(c_lbl)).pack(side="left", padx=(0,15))
    c_lbl = ttk.Label(row1, text="尚未選擇", style="Card.TLabel", foreground="#6B7280")
    c_lbl.pack(side="left")

    row2 = ttk.Frame(card1, style="Card.TFrame")
    row2.pack(fill="x", pady=5)
    ttk.Button(row2, text="選擇 Metadata.json", width=20, command=lambda: select_extract_metadata(m_lbl)).pack(side="left", padx=(0,15))
    m_lbl = ttk.Label(row2, text="尚未選擇", style="Card.TLabel", foreground="#6B7280")
    m_lbl.pack(side="left")

    row3 = ttk.Frame(card1, style="Card.TFrame")
    row3.pack(fill="x", pady=5)
    ttk.Button(row3, text="選擇還原輸出位置", width=20, command=lambda: select_extract_output_folder(o_lbl)).pack(side="left", padx=(0,15))
    o_lbl = ttk.Label(row3, text="尚未選擇", style="Card.TLabel", foreground="#6B7280")
    o_lbl.pack(side="left")

    # 參數設定卡片
    card2 = ttk.Frame(f, style="Card.TFrame", padding=20)
    card2.pack(fill="x", pady=10)

    grid = ttk.Frame(card2, style="Card.TFrame")
    grid.pack(fill="x")
    grid.columnconfigure(1, weight=1)

    ttk.Label(grid, text="解密模式", style="Card.TLabel").grid(row=0, column=0, pady=10, sticky="e", padx=(0,15))
    G["decrypt_mode"] = tk.StringVar(value="自動判斷")
    ttk.Combobox(grid, textvariable=G["decrypt_mode"], values=["自動判斷", "正常解密", "Recovery Key 解密"], state="readonly").grid(row=0, column=1, sticky="we", pady=10)

    ttk.Label(grid, text="加密密碼", style="Card.TLabel").grid(row=1, column=0, pady=10, sticky="e", padx=(0,15))
    G["extract_password"] = tk.StringVar()
    ttk.Entry(grid, textvariable=G["extract_password"], show="●", font=("Arial", 11)).grid(row=1, column=1, sticky="we", pady=10)

    ttk.Label(grid, text="Recovery Key", style="Card.TLabel").grid(row=2, column=0, pady=10, sticky="e", padx=(0,15))
    G["recovery_key_input"] = tk.StringVar()
    ttk.Entry(grid, textvariable=G["recovery_key_input"]).grid(row=2, column=1, sticky="we", pady=10)
    ttk.Label(grid, text="*若使用正常解密模式則不需填寫", style="Card.TLabel", foreground="#9CA3AF", font=("Microsoft JhengHei UI", 9)).grid(row=3, column=1, sticky="w")

    # 底部按鈕
    bottom = ttk.Frame(f)
    bottom.pack(fill="x", pady=20)
    ttk.Button(bottom, text="返回首頁", command=lambda: show("首頁")).pack(side="left")
    ttk.Button(bottom, text="執行解密取出", style="Accent.TButton", command=run_extract).pack(side="right")

    return f


def create_text_page(parent, title, content):
    """通用純文字展示頁面樣板"""
    f = ttk.Frame(parent, padding=30)
    ttk.Label(f, text=title, style="Title.TLabel").pack(anchor="w", pady=(0,20))
    
    # 【修改重點】先宣告並將底部按鈕固定在畫面的最下方
    bottom = ttk.Frame(f)
    bottom.pack(side="bottom", fill="x", pady=(10, 0))
    ttk.Button(bottom, text="返回首頁", command=lambda: show("首頁")).pack(side="left")

    # 再讓中間的文字卡片填滿上方剩餘的所有空間
    card = ttk.Frame(f, style="Card.TFrame", padding=2)
    card.pack(side="top", fill="both", expand=True)

    # 自訂 Text 外觀
    text = tk.Text(
        card, wrap="word", font=("Microsoft JhengHei UI", 11),
        bg="#FFFFFF", fg="#374151", relief="flat", padx=20, pady=20, selectbackground="#DBEAFE"
    )
    text.pack(fill="both", expand=True)
    text.insert("1.0", content)
    text.config(state="disabled")

    return f


def frame_security_settings(parent):
    info = """【 Project Phantasm v2.2 安全模式 】

一、裝置綁定模式

  [開啟]
  解密需要「密碼 + 同一台電腦」。
  系統會根據 Windows MachineGuid、BIOS UUID、BIOS Serial、主機板序號產生裝置指紋。
  如果把 container 和 metadata 複製到其他電腦，正常解密會失敗。

  [關閉]
  解密只需要密碼。適合跨電腦備份或分享給別人。


二、Recovery Key (備援金鑰)

  [產生]
  系統會額外產生 phantasm_recovery_key.txt。
  如果原電腦故障、重灌、主機板更換，可以用「密碼 + Recovery Key」進行緊急解密。

  [不產生]
  如果啟用裝置綁定後原電腦失效，資料可能永久無法解密，請謹慎選擇。


建議預設：
  • 裝置綁定模式：開啟
  • Recovery Key：產生


"""
    return create_text_page(parent, "安全設定說明", info)


def frame_system_info(parent):
    info = """【 系統運作流程 】

[ 加密流程 ]
Secret File 
  ↓ 產生 96 bytes content key material (分成 AES key / position seed / meta key)
  ↓ AES-256-GCM 加密檔案內容
  ↓ SHAKE256 根據 position seed 產生隨機 bit 位置
  ↓ 將 ciphertext bits 分散藏入 random container
  ↓ 用 password / device / recovery 包裝 content key material
輸出：container.bin / metadata.json / recovery_key.txt (若啟用)


[ 解密流程 ]
container + metadata
  ↓ 使用 password + device fingerprint 或 recovery key
  ↓ 解開 content key material，重建 AES key 與 position seed
  ↓ SHAKE256 重建藏入位置，並取出 ciphertext bits
  ↓ AES-GCM decrypt_and_verify
還原原始檔案


【 安全層級架構 】
第一層：Password + PBKDF2 (防止未授權者直接解密)
第二層：Device Fingerprint (讓檔案只能在原電腦正常解密)
第三層：AES-GCM (保護內容機密性，並偵測資料是否被竄改)
第四層：SHAKE256 Scatter (將密文 bit 分散藏入隨機容器，增加隱蔽性)
第五層：Recovery Key (避免原電腦壞掉後完全無法復原)"""
    return create_text_page(parent, "系統架構說明", info)


# ============================================================
# Main
# ============================================================

def main():
    root = tk.Tk()
    root.title("Project Phantasm v2.2 — Clean UI Edition")
    root.geometry("900x700")
    
    setup_theme(root)
    G["root"] = root

    build_menu()

    # 主容器區域
    container = ttk.Frame(root)
    container.pack(fill="both", expand=True)

    G["frames"]["首頁"] = frame_home(container)
    G["frames"]["加密藏入"] = frame_encrypt(container)
    G["frames"]["解密取出"] = frame_extract(container)
    G["frames"]["安全設定說明"] = frame_security_settings(container)
    G["frames"]["系統說明"] = frame_system_info(container)

    for fr in G["frames"].values():
        fr.place(x=0, y=0, relwidth=1, relheight=1)

    G["status"] = tk.StringVar(value="  狀態：初始化完成")

    # 底部狀態列 (套用全新 Style)
    status_bar = ttk.Label(root, textvariable=G["status"], anchor="w", style="Status.TLabel")
    status_bar.pack(fill="x", side="bottom")

    show("首頁")
    root.mainloop()


if __name__ == "__main__":
    main()

In [6]:
"""
╔══════════════════════════════════════════════════════════════╗
║ Project Phantasm v2.2                                      ║
║ Device-Bound Random Container Encryption System             ║
║                                                              ║
║ 版本重點：                                                   ║
║ 1. 移除所有 BitLocker 選項與檢查功能                         ║
║ 2. 保留 AES-256-GCM 加密                                     ║
║ 3. 保留 SHAKE256 隨機 bit 分散藏入                            ║
║ 4. 保留裝置綁定模式                                          ║
║ 5. 保留 Recovery Key 復原機制                                ║
║                                                              ║
║ 依賴：                                                       ║
║ pip install pycryptodome                                     ║
╚══════════════════════════════════════════════════════════════╝
"""

import os
import json
import hashlib
import secrets
import subprocess
import platform
import tkinter as tk
from tkinter import ttk, filedialog, messagebox

from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes


G = {
    "root": None,
    "frames": {},
    "status": None,

    # 加密頁面狀態
    "secret_path": None,
    "output_folder": None,
    "password": None,
    "container_mb": None,
    "device_binding_mode": None,
    "recovery_mode": None,

    # 解密頁面狀態
    "extract_container_path": None,
    "extract_metadata_path": None,
    "extract_output_folder": None,
    "extract_password": None,
    "decrypt_mode": None,
    "recovery_key_input": None,
}


# ============================================================
# UI 基礎控制
# ============================================================

def set_status(msg):
    if G["status"] is not None:
        G["status"].set(msg)


def show(name):
    G["frames"][name].tkraise()
    set_status(f"目前頁面：{name}")


def build_menu():
    mbar = tk.Menu(G["root"])

    m = tk.Menu(mbar, tearoff=0)
    m.add_command(label="首頁", command=lambda: show("首頁"))
    m.add_command(label="加密藏入", command=lambda: show("加密藏入"))
    m.add_command(label="解密取出", command=lambda: show("解密取出"))
    m.add_command(label="安全設定說明", command=lambda: show("安全設定說明"))
    m.add_command(label="系統說明", command=lambda: show("系統說明"))
    m.add_separator()
    m.add_command(label="離開", command=G["root"].quit)

    mbar.add_cascade(label="功能", menu=m)
    G["root"].config(menu=mbar)


# ============================================================
# 基礎工具函式
# ============================================================

def yes_no_value(text):
    return text in ["開啟", "產生", "啟用", "是"]


def safe_filename(name):
    bad = '<>:"/\\|?*'
    out = ""

    for ch in name:
        if ch in bad:
            out += "_"
        else:
            out += ch

    return out


def run_command(cmd, timeout=8):
    """
    執行系統指令，回傳文字。
    失敗時回傳空字串。
    """
    try:
        creationflags = 0

        if os.name == "nt":
            creationflags = subprocess.CREATE_NO_WINDOW

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            errors="replace",
            timeout=timeout,
            creationflags=creationflags
        )

        text = ""

        if result.stdout:
            text += result.stdout

        if result.stderr:
            text += result.stderr

        return text.strip()

    except Exception:
        return ""


def sha256_bytes(data):
    return hashlib.sha256(data).digest()


def sha256_hex(data):
    return hashlib.sha256(data).hexdigest()


# ============================================================
# Device Fingerprint
# ============================================================

def read_windows_machine_guid():
    """
    從 Windows Registry 讀 MachineGuid。
    """
    if os.name != "nt":
        return ""

    try:
        import winreg

        key = winreg.OpenKey(
            winreg.HKEY_LOCAL_MACHINE,
            r"SOFTWARE\Microsoft\Cryptography"
        )

        value, _ = winreg.QueryValueEx(key, "MachineGuid")
        winreg.CloseKey(key)

        return str(value).strip()

    except Exception:
        return ""


def powershell_value(command):
    """
    用 PowerShell 讀取 Windows 裝置資訊。
    """
    if os.name != "nt":
        return ""

    full_cmd = [
        "powershell",
        "-NoProfile",
        "-ExecutionPolicy", "Bypass",
        "-Command",
        command
    ]

    output = run_command(full_cmd)

    lines = []

    for line in output.splitlines():
        line = line.strip()

        if line:
            lines.append(line)

    if not lines:
        return ""

    return lines[0]


def get_bios_uuid():
    return powershell_value("(Get-CimInstance Win32_ComputerSystemProduct).UUID")


def get_bios_serial():
    return powershell_value("(Get-CimInstance Win32_BIOS).SerialNumber")


def get_baseboard_serial():
    return powershell_value("(Get-CimInstance Win32_BaseBoard).SerialNumber")


def get_device_fingerprint_raw():
    """
    取得裝置指紋原始資料。

    使用資料：
    - Windows MachineGuid
    - BIOS UUID
    - BIOS Serial
    - Motherboard Serial

    注意：
    這些資訊可能因為重灌、換主機板、BIOS 設定改變而變化。
    所以裝置綁定模式建議搭配 Recovery Key。
    """
    machine_guid = read_windows_machine_guid()
    bios_uuid = get_bios_uuid()
    bios_serial = get_bios_serial()
    board_serial = get_baseboard_serial()

    parts = {
        "os": platform.system(),
        "machine_guid": machine_guid,
        "bios_uuid": bios_uuid,
        "bios_serial": bios_serial,
        "baseboard_serial": board_serial,
    }

    raw = json.dumps(parts, sort_keys=True, ensure_ascii=False)
    return raw, parts


def get_device_hash():
    raw, parts = get_device_fingerprint_raw()
    return sha256_bytes(raw.encode("utf-8")), parts


# ============================================================
# 密碼學核心
# ============================================================

def bytes_to_bits(data):
    bits = []

    for byte in data:
        for i in range(8):
            bits.append((byte >> i) & 1)

    return bits


def bits_to_bytes(bits):
    out = bytearray()

    for i in range(0, len(bits), 8):
        value = 0
        chunk = bits[i:i + 8]

        for bit_index, bit in enumerate(chunk):
            value |= (bit << bit_index)

        out.append(value)

    return bytes(out)


def set_bit(byte_value, bit_index, secret_bit):
    if secret_bit == 1:
        return byte_value | (1 << bit_index)
    else:
        return byte_value & ~(1 << bit_index)


def get_bit(byte_value, bit_index):
    return (byte_value >> bit_index) & 1


def shake_stream(seed):
    """
    SHAKE256 產生 pseudo-random stream。
    """
    counter = 0

    while True:
        data = seed + counter.to_bytes(8, "big")
        block = hashlib.shake_256(data).digest(64)

        for b in block:
            yield b

        counter += 1


def random_int_from_stream(stream, max_value):
    raw = bytearray()

    for _ in range(8):
        raw.append(next(stream))

    value = int.from_bytes(raw, "big")
    return value % max_value


def generate_positions(seed, total_bytes, bit_count):
    """
    根據 seed 產生不重複 bit 座標。

    global_bit_index = byte_index * 8 + bit_index
    """
    capacity = total_bytes * 8

    if bit_count > capacity:
        raise ValueError(
            "容器容量不夠。\n\n"
            f"需要 bit 數：{bit_count}\n"
            f"容器容量 bit 數：{capacity}\n\n"
            "請增加 container size。"
        )

    stream = shake_stream(seed)
    used = set()
    positions = []

    while len(positions) < bit_count:
        global_bit_index = random_int_from_stream(stream, capacity)

        if global_bit_index in used:
            continue

        used.add(global_bit_index)

        byte_index = global_bit_index // 8
        bit_index = global_bit_index % 8

        positions.append((byte_index, bit_index))

    return positions


def split_content_key_material(key_material):
    """
    96 bytes key material 分成：
    - AES key: 32 bytes
    - position seed: 32 bytes
    - meta key: 32 bytes
    """
    if len(key_material) != 96:
        raise ValueError("key material 長度錯誤，必須是 96 bytes")

    aes_key = key_material[0:32]
    position_seed = key_material[32:64]
    meta_key = key_material[64:96]

    return aes_key, position_seed, meta_key


def derive_access_key(password, salt, access_secret):
    """
    用 password + access_secret + salt 派生包裝 key。

    access_secret 可以是：
    - 裝置 hash
    - 固定值 NO_DEVICE_BINDING
    - recovery key bytes
    """
    if isinstance(access_secret, str):
        access_secret = access_secret.encode("utf-8")

    source = (
        password.encode("utf-8")
        + b"\x00PHANTASM-V2\x00"
        + access_secret
    )

    return hashlib.pbkdf2_hmac(
        "sha256",
        source,
        salt,
        300_000,
        dklen=32
    )


def aes_gcm_encrypt(key, data, aad=None):
    nonce = get_random_bytes(12)
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)

    if aad is not None:
        cipher.update(aad)

    ciphertext, tag = cipher.encrypt_and_digest(data)
    return nonce, ciphertext, tag


def aes_gcm_decrypt(key, nonce, ciphertext, tag, aad=None):
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)

    if aad is not None:
        cipher.update(aad)

    return cipher.decrypt_and_verify(ciphertext, tag)


def wrap_key_material(key_material, password, access_secret, wrap_type):
    """
    將真正用來加密內容的 key_material 包起來。

    好處：
    - 同一份密文可以同時支援正常解密與 Recovery Key 解密
    - 不需要用不同 key 重複加密整份檔案
    """
    salt = get_random_bytes(16)
    access_key = derive_access_key(password, salt, access_secret)

    aad = f"PHANTASM-V2-WRAP-{wrap_type}".encode("utf-8")
    nonce, wrapped, tag = aes_gcm_encrypt(access_key, key_material, aad=aad)

    return {
        "type": wrap_type,
        "salt_hex": salt.hex(),
        "nonce_hex": nonce.hex(),
        "tag_hex": tag.hex(),
        "wrapped_key_material_hex": wrapped.hex()
    }


def unwrap_key_material(wrap_info, password, access_secret):
    wrap_type = wrap_info["type"]

    salt = bytes.fromhex(wrap_info["salt_hex"])
    nonce = bytes.fromhex(wrap_info["nonce_hex"])
    tag = bytes.fromhex(wrap_info["tag_hex"])
    wrapped = bytes.fromhex(wrap_info["wrapped_key_material_hex"])

    access_key = derive_access_key(password, salt, access_secret)
    aad = f"PHANTASM-V2-WRAP-{wrap_type}".encode("utf-8")

    key_material = aes_gcm_decrypt(
        access_key,
        nonce,
        wrapped,
        tag,
        aad=aad
    )

    return key_material


def generate_recovery_key():
    """
    產生人類可保存的 Recovery Key。
    """
    raw = secrets.token_hex(24).upper()

    groups = []

    for i in range(0, len(raw), 4):
        groups.append(raw[i:i + 4])

    return "PHANTASM-RECOVERY-" + "-".join(groups)


def save_recovery_key_file(output_folder, recovery_key):
    path = os.path.join(output_folder, "phantasm_recovery_key.txt")

    content = f"""Project Phantasm v2.2 Recovery Key

請妥善保存這組 Recovery Key。

如果原電腦故障、Windows 重灌、主機板更換，
或無法取得原本的裝置指紋，
可以使用此 Recovery Key 搭配原本密碼解密。

注意：
1. Recovery Key 遺失後無法復原。
2. 請不要和 container.bin、metadata.json 放在同一個不安全位置。
3. 建議離線保存。

Recovery Key:
{recovery_key}
"""

    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

    return path


# ============================================================
# 加密藏入核心
# ============================================================

def encrypt_and_embed_v2(
    secret_path,
    password,
    container_mb,
    output_folder,
    device_binding,
    recovery_enabled
):
    with open(secret_path, "rb") as f:
        secret_data = f.read()

    # 真正用於內容加密與位置散佈的 key material
    content_key_material = get_random_bytes(96)
    aes_key, position_seed, meta_key = split_content_key_material(content_key_material)

    # AES-GCM 加密秘密檔案
    content_nonce, ciphertext, content_tag = aes_gcm_encrypt(
        aes_key,
        secret_data,
        aad=b"PHANTASM-V2-CONTENT"
    )

    # 建立 random container
    container_size = int(container_mb * 1024 * 1024)
    container = bytearray(os.urandom(container_size))

    cipher_bits = bytes_to_bits(ciphertext)

    positions = generate_positions(
        seed=position_seed,
        total_bytes=len(container),
        bit_count=len(cipher_bits)
    )

    for secret_bit, (byte_index, bit_index) in zip(cipher_bits, positions):
        container[byte_index] = set_bit(
            container[byte_index],
            bit_index,
            secret_bit
        )

    # 建立 key wrapping
    wrapped_keys = []

    device_parts = {}
    device_check_salt = get_random_bytes(16)
    device_check_hex = ""

    if device_binding:
        device_hash, device_parts = get_device_hash()
        primary_access_secret = device_hash
        device_check_hex = sha256_hex(device_hash + device_check_salt)
        primary_wrap_type = "primary_device_bound"

    else:
        primary_access_secret = b"NO_DEVICE_BINDING_V1"
        primary_wrap_type = "primary_password_only"

    primary_wrap = wrap_key_material(
        key_material=content_key_material,
        password=password,
        access_secret=primary_access_secret,
        wrap_type=primary_wrap_type
    )

    wrapped_keys.append(primary_wrap)

    recovery_key = None
    recovery_key_path = None
    recovery_check_salt = None
    recovery_check_hex = ""

    if recovery_enabled:
        recovery_key = generate_recovery_key()
        recovery_secret = recovery_key.encode("utf-8")

        recovery_wrap = wrap_key_material(
            key_material=content_key_material,
            password=password,
            access_secret=recovery_secret,
            wrap_type="recovery_key"
        )

        wrapped_keys.append(recovery_wrap)

        recovery_check_salt = get_random_bytes(16)
        recovery_check_hex = sha256_hex(recovery_secret + recovery_check_salt)

        recovery_key_path = save_recovery_key_file(output_folder, recovery_key)

    metadata = {
        "version": "Phantasm-v2.2-No-BitLocker-Device-Bound-Random-Container",
        "created_by": "Project Phantasm v2.2",

        "ciphertext_length": len(ciphertext),
        "container_size": len(container),
        "original_filename": os.path.basename(secret_path),

        "content_nonce_hex": content_nonce.hex(),
        "content_tag_hex": content_tag.hex(),

        "kdf": "PBKDF2-HMAC-SHA256",
        "iterations": 300000,
        "cipher": "AES-256-GCM",
        "position_generator": "SHAKE256",
        "coordinate_type": "(byte_index, bit_index)",

        "device_binding": device_binding,
        "device_fingerprint_method": "MachineGuid+BIOS_UUID+BIOS_Serial+BoardSerial",
        "device_check_salt_hex": device_check_salt.hex(),
        "device_hash_check_hex": device_check_hex,

        "recovery_enabled": recovery_enabled,
        "recovery_check_salt_hex": recovery_check_salt.hex() if recovery_check_salt else "",
        "recovery_key_check_hex": recovery_check_hex,
        "recovery_hint": "Recovery Key was generated at encryption time." if recovery_enabled else "",

        "wrapped_keys": wrapped_keys,

        "security_note": {
            "aes_gcm": "保護內容機密性與完整性",
            "scatter": "密文 bit 被分散藏入 random container",
            "device_binding": "啟用時需要原電腦裝置指紋",
            "recovery_key": "可在原電腦失效時恢復解密能力",
            "bitlocker": "此版本已移除 BitLocker 檢查功能"
        }
    }

    container_path = os.path.join(output_folder, "phantasm_container.bin")
    metadata_path = os.path.join(output_folder, "phantasm_metadata.json")

    with open(container_path, "wb") as f:
        f.write(container)

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=4)

    return {
        "container_path": container_path,
        "metadata_path": metadata_path,
        "recovery_key_path": recovery_key_path,
        "recovery_key": recovery_key
    }


# ============================================================
# 解密取出核心
# ============================================================

def find_wrap(metadata, wrap_type):
    for item in metadata.get("wrapped_keys", []):
        if item.get("type") == wrap_type:
            return item

    return None


def try_unwrap_primary(metadata, password):
    device_binding = metadata.get("device_binding", False)

    if device_binding:
        wrap = find_wrap(metadata, "primary_device_bound")

        if wrap is None:
            raise RuntimeError("metadata 中找不到裝置綁定 key wrap。")

        device_hash, device_parts = get_device_hash()

        check_salt_hex = metadata.get("device_check_salt_hex", "")
        check_hex = metadata.get("device_hash_check_hex", "")

        if check_salt_hex and check_hex:
            check_salt = bytes.fromhex(check_salt_hex)
            current_check = sha256_hex(device_hash + check_salt)

            if current_check != check_hex:
                raise RuntimeError(
                    "目前電腦的裝置指紋與加密時不一致。\n"
                    "如果你已更換電腦或重灌系統，請使用 Recovery Key 解密。"
                )

        return unwrap_key_material(
            wrap_info=wrap,
            password=password,
            access_secret=device_hash
        )

    else:
        wrap = find_wrap(metadata, "primary_password_only")

        if wrap is None:
            raise RuntimeError("metadata 中找不到 password-only key wrap。")

        return unwrap_key_material(
            wrap_info=wrap,
            password=password,
            access_secret=b"NO_DEVICE_BINDING_V1"
        )


def try_unwrap_recovery(metadata, password, recovery_key):
    if not metadata.get("recovery_enabled", False):
        raise RuntimeError("此檔案未啟用 Recovery Key。")

    if not recovery_key:
        raise RuntimeError("請輸入 Recovery Key。")

    wrap = find_wrap(metadata, "recovery_key")

    if wrap is None:
        raise RuntimeError("metadata 中找不到 Recovery Key wrap。")

    recovery_secret = recovery_key.strip().encode("utf-8")

    check_salt_hex = metadata.get("recovery_check_salt_hex", "")
    check_hex = metadata.get("recovery_key_check_hex", "")

    if check_salt_hex and check_hex:
        check_salt = bytes.fromhex(check_salt_hex)
        current_check = sha256_hex(recovery_secret + check_salt)

        if current_check != check_hex:
            raise RuntimeError("Recovery Key 不正確。")

    return unwrap_key_material(
        wrap_info=wrap,
        password=password,
        access_secret=recovery_secret
    )


def extract_and_decrypt_v2(
    container_path,
    metadata_path,
    password,
    output_folder,
    decrypt_mode,
    recovery_key
):
    with open(container_path, "rb") as f:
        container = bytearray(f.read())

    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    version = metadata.get("version", "")

    if not version.startswith("Phantasm-v2"):
        raise RuntimeError("metadata 版本不是 Project Phantasm v2 系列。")

    key_material = None
    errors = []

    if decrypt_mode == "正常解密":
        try:
            key_material = try_unwrap_primary(metadata, password)
        except Exception as e:
            raise RuntimeError(str(e))

    elif decrypt_mode == "Recovery Key 解密":
        try:
            key_material = try_unwrap_recovery(metadata, password, recovery_key)
        except Exception as e:
            raise RuntimeError(str(e))

    else:
        # 自動判斷：先正常解密，再試 Recovery Key
        try:
            key_material = try_unwrap_primary(metadata, password)
        except Exception as e:
            errors.append("正常解密失敗：" + str(e))

            if recovery_key.strip():
                try:
                    key_material = try_unwrap_recovery(metadata, password, recovery_key)
                except Exception as e2:
                    errors.append("Recovery 解密失敗：" + str(e2))

    if key_material is None:
        raise RuntimeError("\n\n".join(errors))

    aes_key, position_seed, meta_key = split_content_key_material(key_material)

    ciphertext_length = int(metadata["ciphertext_length"])
    bit_count = ciphertext_length * 8

    positions = generate_positions(
        seed=position_seed,
        total_bytes=len(container),
        bit_count=bit_count
    )

    extracted_bits = []

    for byte_index, bit_index in positions:
        extracted_bits.append(get_bit(container[byte_index], bit_index))

    ciphertext = bits_to_bytes(extracted_bits)

    content_nonce = bytes.fromhex(metadata["content_nonce_hex"])
    content_tag = bytes.fromhex(metadata["content_tag_hex"])

    plaintext = aes_gcm_decrypt(
        aes_key,
        content_nonce,
        ciphertext,
        content_tag,
        aad=b"PHANTASM-V2-CONTENT"
    )

    original_filename = metadata.get("original_filename", "recovered_file.bin")
    original_filename = safe_filename(original_filename)

    output_path = os.path.join(output_folder, "recovered_" + original_filename)

    with open(output_path, "wb") as f:
        f.write(plaintext)

    return output_path


# ============================================================
# 檔案選擇
# ============================================================

def select_secret_file(label):
    path = filedialog.askopenfilename(title="選擇要加密的檔案")

    if path:
        G["secret_path"] = path
        label.config(text=path)
        set_status("已選擇秘密檔案")


def select_output_folder(label):
    path = filedialog.askdirectory(title="選擇輸出資料夾")

    if path:
        G["output_folder"] = path
        label.config(text=path)
        set_status("已選擇輸出資料夾")


def select_extract_container(label):
    path = filedialog.askopenfilename(
        title="選擇 phantasm_container.bin",
        filetypes=[("BIN files", "*.bin"), ("All files", "*.*")]
    )

    if path:
        G["extract_container_path"] = path
        label.config(text=path)
        set_status("已選擇 container 檔案")


def select_extract_metadata(label):
    path = filedialog.askopenfilename(
        title="選擇 phantasm_metadata.json",
        filetypes=[("JSON files", "*.json"), ("All files", "*.*")]
    )

    if path:
        G["extract_metadata_path"] = path
        label.config(text=path)
        set_status("已選擇 metadata 檔案")


def select_extract_output_folder(label):
    path = filedialog.askdirectory(title="選擇還原輸出資料夾")

    if path:
        G["extract_output_folder"] = path
        label.config(text=path)
        set_status("已選擇還原輸出資料夾")


# ============================================================
# UI 動作
# ============================================================

def run_encrypt():
    if not G["secret_path"]:
        messagebox.showerror("錯誤", "請選擇要加密的檔案")
        return

    if not G["output_folder"]:
        messagebox.showerror("錯誤", "請選擇輸出資料夾")
        return

    password = G["password"].get()

    if not password:
        messagebox.showerror("錯誤", "請輸入密碼")
        return

    try:
        container_mb = float(G["container_mb"].get())

        if container_mb <= 0:
            raise ValueError

    except Exception:
        messagebox.showerror("錯誤", "容器大小必須是正數 MB")
        return

    device_binding = yes_no_value(G["device_binding_mode"].get())
    recovery_enabled = yes_no_value(G["recovery_mode"].get())

    try:
        set_status("正在加密並藏入資料...")

        result = encrypt_and_embed_v2(
            secret_path=G["secret_path"],
            password=password,
            container_mb=container_mb,
            output_folder=G["output_folder"],
            device_binding=device_binding,
            recovery_enabled=recovery_enabled
        )

        msg = (
            "加密藏入完成！\n\n"
            f"容器檔：{result['container_path']}\n"
            f"metadata：{result['metadata_path']}\n"
        )

        if result["recovery_key_path"]:
            msg += f"\nRecovery Key 檔案：{result['recovery_key_path']}\n"
            msg += "\n請務必妥善保存 Recovery Key。"

        messagebox.showinfo("完成", msg)
        set_status("加密藏入完成")

    except Exception as e:
        messagebox.showerror("加密失敗", str(e))
        set_status("加密藏入失敗")


def run_extract():
    if not G["extract_container_path"]:
        messagebox.showerror("錯誤", "請選擇 container.bin")
        return

    if not G["extract_metadata_path"]:
        messagebox.showerror("錯誤", "請選擇 metadata.json")
        return

    if not G["extract_output_folder"]:
        messagebox.showerror("錯誤", "請選擇輸出資料夾")
        return

    password = G["extract_password"].get()

    if not password:
        messagebox.showerror("錯誤", "請輸入密碼")
        return

    decrypt_mode = G["decrypt_mode"].get()
    recovery_key = G["recovery_key_input"].get().strip()

    if decrypt_mode == "Recovery Key 解密" and not recovery_key:
        messagebox.showerror("錯誤", "Recovery Key 解密模式需要輸入 Recovery Key")
        return

    try:
        set_status("正在取出並解密資料...")

        output_path = extract_and_decrypt_v2(
            container_path=G["extract_container_path"],
            metadata_path=G["extract_metadata_path"],
            password=password,
            output_folder=G["extract_output_folder"],
            decrypt_mode=decrypt_mode,
            recovery_key=recovery_key
        )

        messagebox.showinfo(
            "完成",
            f"解密完成！\n\n還原檔案：{output_path}"
        )

        set_status("解密取出完成")

    except Exception as e:
        messagebox.showerror(
            "解密失敗",
            "可能原因：\n"
            "1. 密碼錯誤\n"
            "2. 不是原本的電腦\n"
            "3. Recovery Key 錯誤\n"
            "4. metadata 錯誤\n"
            "5. container 被修改\n\n"
            + str(e)
        )

        set_status("解密取出失敗")


def show_device_fingerprint():
    try:
        device_hash, parts = get_device_hash()

        text = "目前裝置指紋摘要：\n\n"
        text += device_hash.hex()[:32] + "...\n\n"
        text += "使用的裝置資訊：\n"

        for k, v in parts.items():
            text += f"{k}: {v}\n"

        messagebox.showinfo("Device Fingerprint", text)

    except Exception as e:
        messagebox.showerror("錯誤", str(e))


# ============================================================
# UI 分頁
# ============================================================

def frame_home(parent):
    f = ttk.Frame(parent)

    ttk.Label(
        f,
        text="Project Phantasm v2.2",
        font=("Microsoft JhengHei", 24, "bold")
    ).pack(pady=25)

    ttk.Label(
        f,
        text="Device-Bound Random Container Encryption System",
        font=("Microsoft JhengHei", 13)
    ).pack(pady=5)

    ttk.Label(
        f,
        text="AES-GCM + SHAKE256 Scatter + Device Fingerprint + Recovery Key",
        font=("Microsoft JhengHei", 11)
    ).pack(pady=10)

    ttk.Label(
        f,
        text="此版本已移除 BitLocker 選項與檢查功能",
        font=("Microsoft JhengHei", 10)
    ).pack(pady=3)

    ttk.Button(
        f,
        text="開始加密藏入",
        command=lambda: show("加密藏入")
    ).pack(pady=10)

    ttk.Button(
        f,
        text="解密取出",
        command=lambda: show("解密取出")
    ).pack(pady=10)

    ttk.Button(
        f,
        text="安全設定說明",
        command=lambda: show("安全設定說明")
    ).pack(pady=10)

    ttk.Button(
        f,
        text="系統說明",
        command=lambda: show("系統說明")
    ).pack(pady=10)

    return f


def frame_encrypt(parent):
    f = ttk.Frame(parent)

    ttk.Label(
        f,
        text="加密藏入",
        font=("Microsoft JhengHei", 20, "bold")
    ).pack(pady=15)

    file_label = ttk.Label(f, text="尚未選擇檔案")
    file_label.pack(pady=4)

    ttk.Button(
        f,
        text="選擇要加密的檔案",
        command=lambda: select_secret_file(file_label)
    ).pack(pady=4)

    folder_label = ttk.Label(f, text="尚未選擇輸出資料夾")
    folder_label.pack(pady=4)

    ttk.Button(
        f,
        text="選擇輸出資料夾",
        command=lambda: select_output_folder(folder_label)
    ).pack(pady=4)

    form = ttk.Frame(f)
    form.pack(pady=12)

    ttk.Label(form, text="密碼：").grid(row=0, column=0, padx=5, pady=5, sticky="e")
    G["password"] = tk.StringVar()
    ttk.Entry(form, textvariable=G["password"], show="*", width=38).grid(row=0, column=1, padx=5, pady=5)

    ttk.Label(form, text="容器大小 MB：").grid(row=1, column=0, padx=5, pady=5, sticky="e")
    G["container_mb"] = tk.StringVar(value="10")
    ttk.Entry(form, textvariable=G["container_mb"], width=38).grid(row=1, column=1, padx=5, pady=5)

    ttk.Label(form, text="裝置綁定模式：").grid(row=2, column=0, padx=5, pady=5, sticky="e")
    G["device_binding_mode"] = tk.StringVar(value="開啟")
    ttk.Combobox(
        form,
        textvariable=G["device_binding_mode"],
        values=["開啟", "關閉"],
        state="readonly",
        width=35
    ).grid(row=2, column=1, padx=5, pady=5)

    ttk.Label(form, text="Recovery Key：").grid(row=3, column=0, padx=5, pady=5, sticky="e")
    G["recovery_mode"] = tk.StringVar(value="產生")
    ttk.Combobox(
        form,
        textvariable=G["recovery_mode"],
        values=["產生", "不產生"],
        state="readonly",
        width=35
    ).grid(row=3, column=1, padx=5, pady=5)

    buttons = ttk.Frame(f)
    buttons.pack(pady=12)

    ttk.Button(
        buttons,
        text="執行加密藏入",
        command=run_encrypt
    ).grid(row=0, column=0, padx=8)

    ttk.Button(
        buttons,
        text="查看目前裝置指紋",
        command=show_device_fingerprint
    ).grid(row=0, column=1, padx=8)

    ttk.Button(
        f,
        text="回首頁",
        command=lambda: show("首頁")
    ).pack(pady=5)

    return f


def frame_extract(parent):
    f = ttk.Frame(parent)

    ttk.Label(
        f,
        text="解密取出",
        font=("Microsoft JhengHei", 20, "bold")
    ).pack(pady=15)

    container_label = ttk.Label(f, text="尚未選擇 container.bin")
    container_label.pack(pady=4)

    ttk.Button(
        f,
        text="選擇 phantasm_container.bin",
        command=lambda: select_extract_container(container_label)
    ).pack(pady=4)

    metadata_label = ttk.Label(f, text="尚未選擇 metadata.json")
    metadata_label.pack(pady=4)

    ttk.Button(
        f,
        text="選擇 phantasm_metadata.json",
        command=lambda: select_extract_metadata(metadata_label)
    ).pack(pady=4)

    folder_label = ttk.Label(f, text="尚未選擇輸出資料夾")
    folder_label.pack(pady=4)

    ttk.Button(
        f,
        text="選擇還原輸出資料夾",
        command=lambda: select_extract_output_folder(folder_label)
    ).pack(pady=4)

    form = ttk.Frame(f)
    form.pack(pady=12)

    ttk.Label(form, text="解密模式：").grid(row=0, column=0, padx=5, pady=5, sticky="e")
    G["decrypt_mode"] = tk.StringVar(value="自動判斷")
    ttk.Combobox(
        form,
        textvariable=G["decrypt_mode"],
        values=["自動判斷", "正常解密", "Recovery Key 解密"],
        state="readonly",
        width=35
    ).grid(row=0, column=1, padx=5, pady=5)

    ttk.Label(form, text="密碼：").grid(row=1, column=0, padx=5, pady=5, sticky="e")
    G["extract_password"] = tk.StringVar()
    ttk.Entry(
        form,
        textvariable=G["extract_password"],
        show="*",
        width=38
    ).grid(row=1, column=1, padx=5, pady=5)

    ttk.Label(form, text="Recovery Key：").grid(row=2, column=0, padx=5, pady=5, sticky="e")
    G["recovery_key_input"] = tk.StringVar()
    ttk.Entry(
        form,
        textvariable=G["recovery_key_input"],
        width=38
    ).grid(row=2, column=1, padx=5, pady=5)

    ttk.Button(
        f,
        text="執行解密取出",
        command=run_extract
    ).pack(pady=12)

    ttk.Button(
        f,
        text="回首頁",
        command=lambda: show("首頁")
    ).pack(pady=5)

    return f


def frame_security_settings(parent):
    f = ttk.Frame(parent)

    ttk.Label(
        f,
        text="安全設定說明",
        font=("Microsoft JhengHei", 20, "bold")
    ).pack(pady=15)

    text = tk.Text(f, height=24, width=100, wrap="word")
    text.pack(pady=8)

    info = """
Project Phantasm v2.2 安全模式：

一、裝置綁定模式

開啟：
    解密需要「密碼 + 同一台電腦」。
    系統會根據 Windows MachineGuid、BIOS UUID、BIOS Serial、主機板序號產生裝置指紋。
    如果把 container 和 metadata 複製到其他電腦，正常解密會失敗。

關閉：
    解密只需要密碼。
    適合跨電腦備份或分享給別人。


二、Recovery Key

產生：
    系統會額外產生 phantasm_recovery_key.txt。
    如果原電腦故障、重灌、主機板更換，可以用「密碼 + Recovery Key」解密。

不產生：
    如果啟用裝置綁定後原電腦失效，資料可能永久無法解密。


建議預設：
    裝置綁定模式：開啟
    Recovery Key：產生


注意：
    本版本已移除 BitLocker 檢查功能。
    也就是說，它不會檢查輸出磁碟是否受到 BitLocker 保護。
    但 AES-GCM、裝置綁定、Recovery Key、隨機容器藏入功能都仍然保留。
"""

    text.insert("1.0", info)
    text.config(state="disabled")

    ttk.Button(
        f,
        text="回首頁",
        command=lambda: show("首頁")
    ).pack(pady=5)

    return f


def frame_system_info(parent):
    f = ttk.Frame(parent)

    ttk.Label(
        f,
        text="系統說明",
        font=("Microsoft JhengHei", 20, "bold")
    ).pack(pady=15)

    text = tk.Text(f, height=24, width=100, wrap="word")
    text.pack(pady=8)

    info = """
系統流程：

加密流程：

Secret File
  ↓
產生 96 bytes content key material
  ↓
分成 AES key / position seed / meta key
  ↓
AES-256-GCM 加密檔案內容
  ↓
SHAKE256 根據 position seed 產生隨機 bit 位置
  ↓
將 ciphertext bits 分散藏入 random container
  ↓
用 password/device/recovery 包裝 content key material
  ↓
輸出：
    phantasm_container.bin
    phantasm_metadata.json
    phantasm_recovery_key.txt（若啟用）


解密流程：

container + metadata
  ↓
使用 password + device fingerprint 或 password + recovery key
  ↓
解開 content key material
  ↓
重建 AES key / position seed
  ↓
SHAKE256 重建藏入位置
  ↓
取出 ciphertext bits
  ↓
AES-GCM decrypt_and_verify
  ↓
還原檔案


安全層級：

第一層：Password + PBKDF2
    防止未授權者直接解密。

第二層：Device Fingerprint
    讓檔案只能在原電腦正常解密。

第三層：AES-GCM
    保護內容機密性，並偵測資料是否被竄改。

第四層：SHAKE256 Scatter
    將密文 bit 分散藏入 random container，增加資料存在位置的不明顯性。

第五層：Recovery Key
    避免原電腦壞掉後完全無法復原。


注意：
    1. 真正保護內容的是 AES-GCM。
    2. Scatter 提供的是隱蔽性，不是取代 AES。
    3. 裝置綁定會降低可攜性，所以建議搭配 Recovery Key。
    4. 密碼太弱仍然可能被字典攻擊。
    5. Recovery Key 不要和 container、metadata 放在同一個不安全位置。
"""

    text.insert("1.0", info)
    text.config(state="disabled")

    ttk.Button(
        f,
        text="回首頁",
        command=lambda: show("首頁")
    ).pack(pady=5)

    return f


# ============================================================
# main
# ============================================================

def main():
    root = tk.Tk()
    root.title("Project Phantasm v2.2 — No BitLocker Version")
    root.geometry("960x660")
    G["root"] = root

    build_menu()

    container = ttk.Frame(root)
    container.pack(fill="both", expand=True)

    G["frames"]["首頁"] = frame_home(container)
    G["frames"]["加密藏入"] = frame_encrypt(container)
    G["frames"]["解密取出"] = frame_extract(container)
    G["frames"]["安全設定說明"] = frame_security_settings(container)
    G["frames"]["系統說明"] = frame_system_info(container)

    for fr in G["frames"].values():
        fr.place(x=0, y=0, relwidth=1, relheight=1)

    G["status"] = tk.StringVar(value="初始化完成")

    ttk.Label(
        root,
        textvariable=G["status"],
        anchor="w"
    ).pack(fill="x", side="bottom")

    show("首頁")
    root.mainloop()


if __name__ == "__main__":
    main()


In [1]:
"""
╔══════════════════════════════════════════════════════════════╗
║   Project Phantasm Universal AES — 通用檔案保險箱 GUI        ║
║   phantasm_universal_aes_app.py                             ║
║                                                              ║
║  依賴套件：pip install numpy Pillow pycryptodome             ║
║  音訊模組使用標準庫 wave（不需額外安裝）                     ║
║                                                              ║
║  加密輸出：stego.wav + key.png + db_value.txt                ║
║  解密輸入：stego.wav + key.png + db_value.txt + 口令          ║
║                                                              ║
║  安全流程：                                                   ║
║    原始檔案 bytes                                            ║
║      → AES-256-GCM 加密                                      ║
║      → Logistic Map 混沌置亂                                 ║
║      → WAV LSB 隱寫                                          ║
║                                                              ║
║  支援：txt / pdf / docx / png / jpg / mp3 / json / zip / 任意檔案
╚══════════════════════════════════════════════════════════════╝
"""

import os
import wave
import json
import random
import secrets
import hashlib
import threading
import numpy as np
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image

# AES-GCM
from Crypto.Cipher import AES
from Crypto.Protocol.KDF import PBKDF2
from Crypto.Hash import SHA256
from Crypto.Random import get_random_bytes


# ══════════════════════════════════════════════════════════════
#  ███  AES-GCM ENCRYPTION LAYER  ███
#  真正加密層：AES-256-GCM
# ══════════════════════════════════════════════════════════════

AES_PACKAGE_MAGIC = b"PHANTASM_AESGCM_V1"


def derive_aes_key(password: str, salt: bytes) -> bytes:
    """
    使用 PBKDF2-HMAC-SHA256 從口令產生 AES-256 key。

    password + salt → 32 bytes key
    32 bytes = 256 bits，所以是 AES-256。
    """
    return PBKDF2(
        password=password.encode("utf-8"),
        salt=salt,
        dkLen=32,
        count=300_000,
        hmac_hash_module=SHA256
    )


def aes_gcm_encrypt_bytes(data_bytes: bytes, password: str) -> bytes:
    """
    原始 bytes → AES-GCM 加密 package。

    package 格式：
      magic      固定識別字
      salt       16 bytes
      nonce      16 bytes
      tag        16 bytes
      ciphertext 任意長度

    AES-GCM 同時提供：
      1. 機密性：看不懂內容
      2. 完整性：資料被改會驗證失敗
    """
    salt = get_random_bytes(16)
    key = derive_aes_key(password, salt)

    cipher = AES.new(key, AES.MODE_GCM)
    ciphertext, tag = cipher.encrypt_and_digest(data_bytes)
    nonce = cipher.nonce

    package = (
        AES_PACKAGE_MAGIC +
        salt +
        nonce +
        tag +
        ciphertext
    )
    return package


def aes_gcm_decrypt_bytes(package: bytes, password: str) -> bytes:
    """
    AES-GCM package → 原始 bytes。

    如果密碼錯誤、資料損壞、密文被修改，decrypt_and_verify 會失敗。
    """
    magic_len = len(AES_PACKAGE_MAGIC)
    min_len = magic_len + 16 + 16 + 16

    if len(package) < min_len:
        raise ValueError("AES package 長度異常，資料可能損壞。")

    magic = package[:magic_len]
    if magic != AES_PACKAGE_MAGIC:
        raise ValueError("AES package magic 不正確，可能不是本程式產生的 AES-GCM 資料。")

    cursor = magic_len

    salt = package[cursor:cursor + 16]
    cursor += 16

    nonce = package[cursor:cursor + 16]
    cursor += 16

    tag = package[cursor:cursor + 16]
    cursor += 16

    ciphertext = package[cursor:]

    key = derive_aes_key(password, salt)
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)

    try:
        plaintext = cipher.decrypt_and_verify(ciphertext, tag)
    except ValueError:
        raise ValueError("AES-GCM 驗證失敗：密碼錯誤，或資料已被竄改 / 損壞。")

    return plaintext


# ══════════════════════════════════════════════════════════════
#  ███  CORE LOGIC  ███
#  Logistic Map + bytes 混沌置亂
# ══════════════════════════════════════════════════════════════

def generate_logistic_sequence(size: int, x0: float, r: float = 3.99) -> np.ndarray:
    """
    Logistic Map：
      x_{n+1} = r * x_n * (1 - x_n)

    r=3.99 時通常位於混沌區。
    這裡用來產生排序序列，進而打亂 bytes 位置。
    """
    x = x0
    seq = np.zeros(size, dtype=np.float64)

    for i in range(size):
        x = r * x * (1 - x)
        seq[i] = x

    return seq


def encrypt_bytes_to_noise(data_bytes: bytes, x0: float) -> np.ndarray:
    """
    bytes → noise bytes。

    原理：
      1. 用 logistic map 產生 chaos_seq
      2. 對 chaos_seq argsort 取得排列索引
      3. 用排列索引打亂 byte 順序

    注意：
      這是混沌置亂，不是正式加密。
      真正加密已經由 AES-GCM 負責。
    """
    arr = np.frombuffer(data_bytes, dtype=np.uint8).copy()
    chaos_seq = generate_logistic_sequence(len(arr), x0)
    shuffle_indices = np.argsort(chaos_seq)
    encrypted = arr[shuffle_indices]
    return encrypted


def decrypt_noise_to_bytes(noise_array: np.ndarray, x0: float) -> bytes:
    """
    noise bytes → 原始 bytes。

    使用同一個 x0 產生相同排列，再做逆映射。
    """
    flat = noise_array.flatten()
    chaos_seq = generate_logistic_sequence(len(flat), x0)
    shuffle_indices = np.argsort(chaos_seq)

    recovered = np.zeros(len(flat), dtype=np.uint8)
    recovered[shuffle_indices] = flat

    return recovered.tobytes()


# ══════════════════════════════════════════════════════════════
#  ███  FILE METADATA  ███
#  檔名 / 副檔名 / 大小資訊
# ══════════════════════════════════════════════════════════════

def build_metadata(secret_file_path: str) -> bytes:
    """
    建立檔案 metadata。

    解密時會用 metadata 還原：
      1. 原始檔名
      2. 原始大小
      3. 系統格式版本
    """
    filename = os.path.basename(secret_file_path)
    file_size = os.path.getsize(secret_file_path)

    meta = {
        "filename": filename,
        "file_size": file_size,
        "format": "PHANTASM_UNIVERSAL_AES_V1",
        "encryption": "AES-256-GCM",
        "obfuscation": "LogisticMap",
        "steganography": "WAV_LSB"
    }

    return json.dumps(meta, ensure_ascii=False).encode("utf-8")


def parse_metadata(meta_bytes: bytes) -> dict:
    """解析 metadata JSON。"""
    return json.loads(meta_bytes.decode("utf-8"))


def safe_output_path(output_dir: str, filename: str) -> str:
    """
    避免解密輸出覆蓋既有檔案。
    如果檔名已存在，會自動加 recovered_1、recovered_2...
    """
    base, ext = os.path.splitext(filename)
    candidate = os.path.join(output_dir, filename)

    count = 1
    while os.path.exists(candidate):
        candidate = os.path.join(output_dir, f"{base}_recovered_{count}{ext}")
        count += 1

    return candidate


# ══════════════════════════════════════════════════════════════
#  ███  MEDIA PNG  ███
#  LSB 金鑰隱寫：隱藏 / 提取 x0 浮點數
# ══════════════════════════════════════════════════════════════

def _get_salt(password: str) -> int:
    """
    口令 → SHA-256 → 取前 8 hex 位作為整數鹽值。

    這裡不是 AES 的 salt。
    這裡是用來保護 key.png 中 x0 的 offset。
    """
    h = hashlib.sha256(password.encode("utf-8")).hexdigest()
    return int(h[:8], 16)


def hide_key_to_png(png_path: str, key_float: float, password: str, output_path: str) -> int:
    """
    把 logistic map 初始值 x0 用 LSB 隱寫進 PNG。

    x0 使用 IEEE 754 double，完整 64-bit 儲存。
    db_value = real_offset XOR SHA256(password)[:8]

    解密時需要：
      key.png + db_value.txt + password
    """
    import struct

    key_bytes = struct.pack(">d", key_float)
    key_bits = "".join(f"{b:08b}" for b in key_bytes)

    img = Image.open(png_path).convert("RGB")
    img_arr = np.array(img, dtype=np.uint8)

    height, width = img_arr.shape[:2]
    total_pixels = width * height

    n_bits = len(key_bits)
    step = 3
    needed = n_bits * step + 1
    max_safe = total_pixels - needed

    if max_safe < 0:
        raise ValueError(
            f"PNG 圖片太小（僅 {total_pixels} 像素），需要至少 {needed} 像素。"
        )

    real_offset = random.randint(0, min(5000, max_safe))
    salt = _get_salt(password)
    db_value = real_offset ^ salt

    for i, bit in enumerate(key_bits):
        idx = real_offset + i * step
        row, col = idx // width, idx % width
        img_arr[row, col, 0] = (img_arr[row, col, 0] & 0xFE) | int(bit)

    Image.fromarray(img_arr).save(output_path)
    return db_value


def extract_key_from_png(png_path: str, db_value: int, password: str) -> float:
    """
    從 PNG 還原 x0。
    """
    import struct

    salt = _get_salt(password)
    real_offset = db_value ^ salt

    img = Image.open(png_path).convert("RGB")
    img_arr = np.array(img, dtype=np.uint8)

    height, width = img_arr.shape[:2]
    total_pixels = width * height

    n_bits = 64
    step = 3
    bits = []

    for i in range(n_bits):
        idx = real_offset + i * step

        if idx >= total_pixels:
            raise ValueError(
                "PNG 金鑰提取失敗：像素索引超出圖片範圍。\n"
                "請確認 key.png、db_value.txt 和口令三者來自同一次加密。"
            )

        row, col = idx // width, idx % width
        bits.append(str(img_arr[row, col, 0] & 1))

    bit_str = "".join(bits)
    raw_bytes = bytes(int(bit_str[i:i + 8], 2) for i in range(0, 64, 8))

    return struct.unpack(">d", raw_bytes)[0]


# ══════════════════════════════════════════════════════════════
#  ███  MEDIA WAV  ███
#  LSB 通用 bytes 隱寫進 WAV 音訊
# ══════════════════════════════════════════════════════════════

def hide_bytes_to_wav(wav_path: str, encrypted_bytes: np.ndarray, meta_bytes: bytes, save_path: str) -> tuple:
    """
    把任意 bytes LSB 藏進 WAV 音訊。

    資料格式：
      magic: b"PHANTASM_UAES1"
      metadata 長度：4 bytes
      metadata JSON bytes
      encrypted data 長度：8 bytes
      encrypted data bytes
      end marker: b"PHANTASM_END"
    """
    try:
        magic = b"PHANTASM_UAES1"
        end_marker = b"PHANTASM_END"

        data_bytes = encrypted_bytes.astype(np.uint8).tobytes()

        payload = (
            magic +
            len(meta_bytes).to_bytes(4, "big") +
            meta_bytes +
            len(data_bytes).to_bytes(8, "big") +
            data_bytes +
            end_marker
        )

        binary_msg = "".join(format(b, "08b") for b in payload)

        with wave.open(wav_path, "rb") as audio:
            params = audio.getparams()
            frame_bytes = bytearray(audio.readframes(audio.getnframes()))

        if len(binary_msg) > len(frame_bytes):
            needed_kb = len(binary_msg) / 8 / 1024
            avail_kb = len(frame_bytes) / 1024

            return False, (
                f"WAV 容量不足！\n"
                f"需要約 {needed_kb:.1f} KB，WAV 僅有 {avail_kb:.1f} KB。\n"
                f"請換更長的 WAV，或縮小要隱藏的檔案。"
            )

        for i, bit in enumerate(binary_msg):
            frame_bytes[i] = (frame_bytes[i] & 0xFE) | int(bit)

        with wave.open(save_path, "wb") as f:
            f.setparams(params)
            f.writeframes(bytes(frame_bytes))

        return True, save_path

    except Exception as e:
        return False, str(e)


def extract_bytes_from_wav(wav_path: str) -> tuple:
    """
    從 WAV 提取 metadata 與加密資料。

    回傳：
      (encrypted_array, metadata_dict)

    失敗：
      (None, None)
    """
    try:
        magic_expected = b"PHANTASM_UAES1"
        end_marker_expected = b"PHANTASM_END"

        with wave.open(wav_path, "rb") as audio:
            frame_bytes = bytearray(audio.readframes(audio.getnframes()))

        def get_byte(start_bit: int) -> int:
            bits = "".join(str(frame_bytes[j] & 1) for j in range(start_bit, start_bit + 8))
            return int(bits, 2)

        cursor = 0

        def read_bytes(n: int) -> bytes:
            nonlocal cursor
            out = bytes(get_byte(i) for i in range(cursor, cursor + n * 8, 8))
            cursor += n * 8
            return out

        magic = read_bytes(len(magic_expected))
        if magic != magic_expected:
            raise ValueError("這不是 Phantasm Universal AES 產生的 WAV，或檔案已損壞。")

        meta_len = int.from_bytes(read_bytes(4), "big")
        meta_bytes = read_bytes(meta_len)
        metadata = parse_metadata(meta_bytes)

        data_len = int.from_bytes(read_bytes(8), "big")
        encrypted_data = read_bytes(data_len)

        end_marker = read_bytes(len(end_marker_expected))
        if end_marker != end_marker_expected:
            raise ValueError("結束標記不正確，可能資料被截斷或 WAV 已損壞。")

        arr = np.frombuffer(encrypted_data, dtype=np.uint8)
        return arr, metadata

    except Exception:
        return None, None


# ══════════════════════════════════════════════════════════════
#  ███  KEY FILE  ███
#  x0 自動產生 + db_value txt 存取
# ══════════════════════════════════════════════════════════════

def generate_x0() -> float:
    """
    用密碼學安全亂數產生 logistic map 的 x0。

    避開太接近 0 或 1 的值，降低 logistic map 退化機率。
    """
    while True:
        x0 = secrets.randbelow(10**15) / 10**15

        if 0.02 < x0 < 0.98:
            return x0


def save_db_value(db_val: int, x0: float, txt_path: str):
    """
    將 db_value 寫成 txt。
    x0_ref 只給除錯用，不建議正式版本保留。
    """
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("# Project Phantasm Universal AES — 解密金鑰檔案\n")
        f.write("# 解密時需要：stego.wav + key.png + db_value.txt + 口令\n")
        f.write("# 系統流程：AES-256-GCM → Logistic Map → WAV LSB\n\n")
        f.write(f"db_value={db_val}\n")
        f.write("# x0_ref 僅供除錯參考；正式版本可以刪除這行\n")
        f.write(f"x0_ref={x0:.15f}\n")


def load_db_value_from_txt(txt_path: str) -> int:
    """
    從 txt 讀回 db_value。
    """
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line.startswith("db_value="):
                return int(line.split("=", 1)[1])

    raise ValueError("找不到 db_value 欄位，請確認是 Phantasm Universal AES 產生的 txt 檔案。")


# ══════════════════════════════════════════════════════════════
#  ███  INTEGRATOR  ███
#  完整加密 / 解密流程
# ══════════════════════════════════════════════════════════════

def full_encrypt(secret_file: str, cover_wav: str, key_png: str,
                 password: str, out_wav: str, out_key_png: str, out_txt: str,
                 progress_cb=None) -> tuple:
    """
    完整加密流程：

      1. 讀取任意檔案 bytes
      2. 使用 AES-256-GCM 加密檔案內容
      3. 產生 logistic map 初始值 x0
      4. 用 logistic map 打亂 AES package
      5. 將置亂後 bytes 藏進 WAV
      6. 將 x0 藏進 PNG，db_value 存 txt
    """
    try:
        def step(msg):
            if progress_cb:
                progress_cb(msg)

        step("【準備】讀取秘密檔案 bytes...")
        with open(secret_file, "rb") as f:
            data_bytes = f.read()

        meta_bytes = build_metadata(secret_file)

        step(f"  檔案：{os.path.basename(secret_file)}")
        step(f"  原始大小：{len(data_bytes):,} bytes")

        step("【步驟 1/5】使用 AES-256-GCM 加密檔案內容...")
        aes_package = aes_gcm_encrypt_bytes(data_bytes, password)
        step(f"  AES package 大小：{len(aes_package):,} bytes")
        step("  AES-GCM 已產生 salt / nonce / tag")

        step("【步驟 2/5】自動產生 logistic map 金鑰 x0...")
        x0 = generate_x0()
        step("  x0 已產生，每次加密唯一")

        step("【步驟 3/5】用 logistic map 打亂 AES package...")
        encrypted_arr = encrypt_bytes_to_noise(aes_package, x0)
        step(f"  已混沌置亂 {len(encrypted_arr):,} bytes")

        step("【步驟 4/5】將置亂後 bytes LSB 隱寫進 WAV...")
        ok, result = hide_bytes_to_wav(cover_wav, encrypted_arr, meta_bytes, out_wav)

        if not ok:
            return False, result, 0.0

        step(f"  輸出 WAV：{os.path.basename(out_wav)}")

        step("【步驟 5/5】將 x0 LSB 隱寫進 PNG，db_value 存成 txt...")
        db_val = hide_key_to_png(key_png, x0, password, out_key_png)
        save_db_value(db_val, x0, out_txt)

        step(f"  輸出 PNG：{os.path.basename(out_key_png)}")
        step(f"  db_value 已存入：{os.path.basename(out_txt)}")
        step("\n✅ 加密完成！")
        step("  系統流程：AES-256-GCM → logistic map → WAV LSB")

        return True, str(db_val), x0

    except Exception as e:
        return False, str(e), 0.0


def full_decrypt(stego_wav: str, key_png: str, password: str,
                 db_value: int, output_dir: str, progress_cb=None) -> tuple:
    """
    完整解密流程：

      1. 從 PNG + db_value + 口令還原 x0
      2. 從 WAV 提取置亂後 AES package
      3. 用 logistic map 逆映射還原 AES package
      4. 使用 AES-256-GCM 解密與驗證
      5. 按原檔名輸出
    """
    try:
        def step(msg):
            if progress_cb:
                progress_cb(msg)

        step("【步驟 1/4】從 PNG 提取 logistic map 金鑰 x0...")
        x0 = extract_key_from_png(key_png, db_value, password)

        if not (0.0 < x0 < 1.0):
            return False, "x0 還原值異常，請確認 key.png、db_value.txt 和口令正確。"

        step("  x0 已還原，值域驗證通過")

        step("【步驟 2/4】從 WAV 提取置亂後 AES package 與 metadata...")
        encrypted_arr, metadata = extract_bytes_from_wav(stego_wav)

        if encrypted_arr is None:
            return False, "從 WAV 提取資料失敗，可能 WAV 不是本程式產生或檔案已損壞。"

        step(f"  原始檔名：{metadata.get('filename', 'unknown.bin')}")
        step(f"  提取資料大小：{len(encrypted_arr):,} bytes")

        step("【步驟 3/4】用 logistic map 逆映射還原 AES package...")
        aes_package = decrypt_noise_to_bytes(encrypted_arr, x0)
        step(f"  AES package 還原完成：{len(aes_package):,} bytes")

        step("【步驟 4/4】使用 AES-256-GCM 解密與完整性驗證...")
        recovered_bytes = aes_gcm_decrypt_bytes(aes_package, password)
        step(f"  AES-GCM 驗證通過，還原大小：{len(recovered_bytes):,} bytes")

        expected_size = metadata.get("file_size")
        if expected_size is not None and len(recovered_bytes) != expected_size:
            return False, (
                "還原後檔案大小不符合 metadata。\n"
                "可能是密碼、key.png、db_value.txt 或 stego.wav 不匹配。"
            )

        filename = metadata.get("filename", "recovered_file.bin")
        out_path = safe_output_path(output_dir, filename)

        with open(out_path, "wb") as f:
            f.write(recovered_bytes)

        step(f"  輸出：{out_path}")
        step("\n✅ 解密完成！")

        return True, out_path

    except Exception as e:
        return False, str(e)


# ══════════════════════════════════════════════════════════════
#  ███  GUI STYLE  ███
# ══════════════════════════════════════════════════════════════

C = {
    "bg": "#0d1117",
    "surface": "#161b22",
    "border": "#30363d",
    "accent": "#58a6ff",
    "accent2": "#3fb950",
    "accent3": "#f85149",
    "accent4": "#d2a8ff",
    "text": "#e6edf3",
    "subtext": "#8b949e",
    "entry_bg": "#21262d",
}

FONT_TITLE = ("Consolas", 22, "bold")
FONT_HEAD = ("Consolas", 13, "bold")
FONT_BODY = ("Microsoft JhengHei", 11)
FONT_MONO = ("Consolas", 10)
FONT_STATUS = ("Consolas", 10)


# ══════════════════════════════════════════════════════════════
#  ███  GUI APP  ███
# ══════════════════════════════════════════════════════════════

class PhantasmUniversalAESApp(tk.Tk):
    def __init__(self):
        super().__init__()

        self.title("Project Phantasm Universal AES — 通用檔案保險箱")
        self.geometry("1020x740")
        self.minsize(900, 660)
        self.configure(bg=C["bg"])
        self.resizable(True, True)

        self.status_var = tk.StringVar(value="就緒")
        self.db_value = tk.StringVar(value="")

        self._build_title_bar()
        self._build_notebook()
        self._build_status_bar()

    def _build_title_bar(self):
        bar = tk.Frame(self, bg=C["surface"], height=58)
        bar.pack(fill="x")
        bar.pack_propagate(False)

        tk.Label(
            bar,
            text="⬡  Project Phantasm Universal AES",
            font=FONT_TITLE,
            fg=C["accent"],
            bg=C["surface"]
        ).pack(side="left", padx=20, pady=8)

        tk.Label(
            bar,
            text="AES-256-GCM + Logistic Map + WAV LSB",
            font=FONT_BODY,
            fg=C["subtext"],
            bg=C["surface"]
        ).pack(side="left", padx=4)

        tk.Frame(self, bg=C["border"], height=1).pack(fill="x")

    def _build_notebook(self):
        style = ttk.Style(self)
        style.theme_use("clam")

        style.configure(
            "P.TNotebook",
            background=C["bg"],
            borderwidth=0,
            tabmargins=[0, 4, 0, 0]
        )

        style.configure(
            "P.TNotebook.Tab",
            background=C["surface"],
            foreground=C["subtext"],
            font=FONT_HEAD,
            padding=[18, 8],
            borderwidth=0
        )

        style.map(
            "P.TNotebook.Tab",
            background=[("selected", C["bg"])],
            foreground=[("selected", C["accent"])]
        )

        nb = ttk.Notebook(self, style="P.TNotebook")
        nb.pack(fill="both", expand=True)

        enc = tk.Frame(nb, bg=C["bg"])
        dec = tk.Frame(nb, bg=C["bg"])
        abt = tk.Frame(nb, bg=C["bg"])

        nb.add(enc, text="  🔒  加密  ")
        nb.add(dec, text="  🔓  解密  ")
        nb.add(abt, text="  ℹ  關於  ")

        self._build_encrypt_page(enc)
        self._build_decrypt_page(dec)
        self._build_about_page(abt)

    def _build_status_bar(self):
        tk.Frame(self, bg=C["border"], height=1).pack(fill="x")

        bar = tk.Frame(self, bg=C["surface"], height=28)
        bar.pack(fill="x", side="bottom")
        bar.pack_propagate(False)

        tk.Label(
            bar,
            text="●",
            fg=C["accent2"],
            bg=C["surface"],
            font=("Consolas", 10)
        ).pack(side="left", padx=8)

        tk.Label(
            bar,
            textvariable=self.status_var,
            fg=C["subtext"],
            bg=C["surface"],
            font=FONT_STATUS,
            anchor="w"
        ).pack(side="left")

    def set_status(self, msg: str):
        self.status_var.set(msg)
        self.update_idletasks()

    # ══════════════════════════════════════════════════════════
    #  加密頁面
    # ══════════════════════════════════════════════════════════

    def _build_encrypt_page(self, parent):
        left = tk.Frame(parent, bg=C["bg"])
        right = tk.Frame(parent, bg=C["bg"])

        left.pack(side="left", fill="both", expand=True, padx=(20, 8), pady=20)
        right.pack(side="right", fill="both", expand=True, padx=(8, 20), pady=20)

        self._section(left, "輸入設定", C["accent"])
        self.enc_secret = self._file_row(left, "📦  要隱藏的任意檔案", "選擇檔案", [("所有檔案", "*.*")])
        self.enc_wav = self._file_row(left, "🎵  封面 WAV 音訊（藏檔案 bytes 用）", "選擇 WAV", [("WAV 音訊", "*.wav")])
        self.enc_keypng = self._file_row(left, "🔑  金鑰載體 PNG（藏 x0 用）", "選擇 PNG", [("PNG 圖片", "*.png")])

        self._label(left, "保護口令")
        self.enc_pw = self._entry(left, show="●")

        self._section(left, "輸出設定（三個檔案）", C["accent"])
        self.enc_out_wav = self._save_row(left, "💾  輸出 WAV", "stego_output.wav", [("WAV 音訊", "*.wav")])
        self.enc_out_png = self._save_row(left, "💾  輸出金鑰 PNG", "key_output.png", [("PNG 圖片", "*.png")])
        self.enc_out_txt = self._save_row(left, "💾  db_value.txt", "db_value.txt", [("文字檔案", "*.txt")])

        self._section(right, "執行加密", C["accent"])

        hint = tk.Frame(right, bg=C["surface"], highlightbackground=C["border"], highlightthickness=1)
        hint.pack(fill="x", pady=(0, 8))

        tk.Label(
            hint,
            text=(
                "📦 支援任意檔案：txt / pdf / docx / png / jpg / json / mp3 / zip...\n"
                "🔐 加密流程：AES-256-GCM → Logistic Map → WAV LSB"
            ),
            fg=C["accent2"],
            bg=C["surface"],
            font=FONT_MONO,
            wraplength=450,
            justify="left"
        ).pack(anchor="w", padx=10, pady=8)

        self.enc_log = self._log_box(right)

        dv_frame = tk.Frame(right, bg=C["surface"], highlightbackground=C["border"], highlightthickness=1)
        dv_frame.pack(fill="x", pady=(8, 0))

        tk.Label(
            dv_frame,
            text="🗝  db_value（已自動存入 txt）",
            fg=C["accent"],
            bg=C["surface"],
            font=FONT_MONO
        ).pack(anchor="w", padx=10, pady=(8, 2))

        dv_inner = tk.Frame(dv_frame, bg=C["surface"])
        dv_inner.pack(fill="x", padx=10, pady=(0, 8))

        self.dv_display = tk.Entry(
            dv_inner,
            textvariable=self.db_value,
            state="readonly",
            font=FONT_MONO,
            bg=C["entry_bg"],
            fg=C["accent2"],
            readonlybackground=C["entry_bg"],
            relief="flat",
            bd=0
        )
        self.dv_display.pack(side="left", fill="x", expand=True, ipady=5, padx=(0, 8))

        tk.Button(
            dv_inner,
            text="複製",
            font=FONT_MONO,
            bg=C["border"],
            fg=C["text"],
            relief="flat",
            activebackground=C["accent"],
            activeforeground="white",
            cursor="hand2",
            command=self._copy_dbval
        ).pack(side="right")

        self._big_button(right, "▶  開始加密", C["accent"], self._run_encrypt)

    # ══════════════════════════════════════════════════════════
    #  解密頁面
    # ══════════════════════════════════════════════════════════

    def _build_decrypt_page(self, parent):
        left = tk.Frame(parent, bg=C["bg"])
        right = tk.Frame(parent, bg=C["bg"])

        left.pack(side="left", fill="both", expand=True, padx=(20, 8), pady=20)
        right.pack(side="right", fill="both", expand=True, padx=(8, 20), pady=20)

        self._section(left, "輸入設定", C["accent4"])
        self.dec_wav = self._file_row(left, "🎵  含密文的 WAV", "選擇 WAV", [("WAV 音訊", "*.wav")])
        self.dec_keypng = self._file_row(left, "🔑  含金鑰的 PNG", "選擇 PNG", [("PNG 圖片", "*.png")])

        self._label(left, "📄  db_value.txt → 選擇後自動填入 db_value")

        dbtxt_row = tk.Frame(left, bg=C["bg"])
        dbtxt_row.pack(fill="x", pady=(0, 2))

        self.dec_txt_var = tk.StringVar()

        tk.Entry(
            dbtxt_row,
            textvariable=self.dec_txt_var,
            font=FONT_MONO,
            bg=C["entry_bg"],
            fg=C["text"],
            insertbackground=C["text"],
            relief="flat",
            bd=0
        ).pack(side="left", fill="x", expand=True, ipady=6, ipadx=8)

        tk.Button(
            dbtxt_row,
            text="選擇 txt",
            font=FONT_MONO,
            bg=C["border"],
            fg=C["text"],
            relief="flat",
            activebackground=C["accent4"],
            activeforeground="white",
            cursor="hand2",
            padx=10,
            command=self._load_txt_file
        ).pack(side="right", padx=(6, 0))

        tk.Frame(left, bg=C["border"], height=1).pack(fill="x")

        self._label(left, "db_value 數字")
        self.dec_dbval = self._entry(left)

        self._label(left, "口令")
        self.dec_pw = self._entry(left, show="●")

        self._section(left, "輸出設定", C["accent4"])
        self.dec_out_dir = self._dir_row(left, "📁  還原檔案輸出資料夾")

        self._section(right, "執行解密", C["accent4"])
        self.dec_log = self._log_box(right)

        info_frame = tk.Frame(right, bg=C["surface"], highlightbackground=C["border"], highlightthickness=1)
        info_frame.pack(fill="both", expand=True, pady=(8, 0))

        tk.Label(
            info_frame,
            text="📦  還原資訊",
            fg=C["accent4"],
            bg=C["surface"],
            font=FONT_MONO
        ).pack(anchor="w", padx=10, pady=(8, 4))

        self.recovered_info = tk.Label(
            info_frame,
            bg=C["entry_bg"],
            text="解密後會顯示輸出檔案路徑",
            fg=C["subtext"],
            font=FONT_BODY,
            wraplength=450,
            justify="left"
        )
        self.recovered_info.pack(fill="both", expand=True, padx=10, pady=(0, 10))

        self._big_button(right, "▶  開始解密", C["accent4"], self._run_decrypt)

    # ══════════════════════════════════════════════════════════
    #  關於頁面
    # ══════════════════════════════════════════════════════════

    def _build_about_page(self, parent):
        canvas = tk.Canvas(parent, bg=C["bg"], highlightthickness=0)
        sb = ttk.Scrollbar(parent, orient="vertical", command=canvas.yview)
        canvas.configure(yscrollcommand=sb.set)

        sb.pack(side="right", fill="y")
        canvas.pack(side="left", fill="both", expand=True)

        frame = tk.Frame(canvas, bg=C["bg"])
        canvas.create_window((0, 0), window=frame, anchor="nw")
        frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))

        def lbl(text, font, fg, pady=(2, 2), padx=0):
            tk.Label(
                frame,
                text=text,
                font=font,
                fg=fg,
                bg=C["bg"],
                justify="left",
                wraplength=900
            ).pack(anchor="w", pady=pady, padx=(40 + padx, 40))

        lbl("Project Phantasm Universal AES", ("Consolas", 26, "bold"), C["accent"], (20, 2))
        lbl(
            "通用檔案版本：任何檔案最終都可以視為 bytes，因此可以被統一加密與隱寫。",
            FONT_BODY,
            C["subtext"],
            (2, 16)
        )

        lbl("🔒 加密流程", FONT_HEAD, C["accent2"], (14, 4))
        lbl(
            "1. 讀取任意檔案 bytes\n"
            "2. 使用 AES-256-GCM 加密檔案內容\n"
            "3. 使用 Logistic Map 打亂 AES package\n"
            "4. 把置亂後 bytes 藏入 WAV LSB\n"
            "5. 把 x0 金鑰藏入 PNG LSB\n"
            "6. db_value 存入 txt",
            FONT_BODY,
            C["text"],
            (0, 10),
            8
        )

        lbl("🔓 解密流程", FONT_HEAD, C["accent4"], (14, 4))
        lbl(
            "1. 使用 key.png + db_value.txt + 口令還原 x0\n"
            "2. 從 WAV 提取置亂後 AES package\n"
            "3. 用 x0 做 logistic map 逆映射\n"
            "4. 使用 AES-256-GCM 解密並驗證 tag\n"
            "5. 依 metadata 還原原始檔名與內容",
            FONT_BODY,
            C["text"],
            (0, 10),
            8
        )

        lbl("🧠 技術定位", FONT_HEAD, C["accent"], (14, 4))
        lbl(
            "AES-256-GCM：真正負責檔案內容加密與完整性驗證。\n"
            "Logistic Map：負責混沌置亂，讓提取後資料不容易直接還原。\n"
            "WAV LSB：負責把置亂後資料隱藏在音訊中。\n"
            "PNG LSB：負責隱藏 logistic map 的 x0。",
            FONT_BODY,
            C["text"],
            (0, 10),
            8
        )

        lbl("⚠ 注意", FONT_HEAD, C["accent3"], (14, 4))
        lbl(
            "這仍是專題展示與原型系統。若要正式企業使用，還應加入："
            "更完整的金鑰管理、檔案簽章、錯誤更正碼、載體品質檢查、操作紀錄、權限控管與安全審計。",
            FONT_BODY,
            C["text"],
            (0, 10),
            8
        )

    # ══════════════════════════════════════════════════════════
    #  執行加密 / 解密
    # ══════════════════════════════════════════════════════════

    def _run_encrypt(self):
        secret = self.enc_secret.get().strip()
        wav = self.enc_wav.get().strip()
        keypng = self.enc_keypng.get().strip()
        pw = self.enc_pw.get().strip()
        o_wav = self.enc_out_wav.get().strip()
        o_png = self.enc_out_png.get().strip()
        o_txt = self.enc_out_txt.get().strip()

        errors = []

        if not os.path.exists(secret):
            errors.append("找不到要隱藏的檔案")
        if not os.path.exists(wav):
            errors.append("找不到封面 WAV 音訊")
        if not os.path.exists(keypng):
            errors.append("找不到金鑰載體 PNG")
        if not pw:
            errors.append("口令不能為空")
        if not o_wav:
            errors.append("請設定輸出 WAV 路徑")
        if not o_png:
            errors.append("請設定輸出金鑰 PNG 路徑")
        if not o_txt:
            errors.append("請設定 db_value txt 路徑")

        if errors:
            messagebox.showerror("輸入錯誤", "\n".join(f"• {e}" for e in errors))
            return

        self._log_clear(self.enc_log)
        self.db_value.set("")

        def task():
            ok, result, x0 = full_encrypt(
                secret,
                wav,
                keypng,
                pw,
                o_wav,
                o_png,
                o_txt,
                progress_cb=lambda m: self._log_append(self.enc_log, m)
            )

            if ok:
                self.db_value.set(result)
                self._log_append(
                    self.enc_log,
                    f"\n🗝  db_value = {result}"
                    f"\n📄  已自動存入：{o_txt}"
                    f"\n⚠  解密需要 stego.wav + key.png + db_value.txt + 口令"
                )
                self.set_status("加密完成")
            else:
                self._log_append(self.enc_log, f"\n❌ 加密失敗：{result}")
                self.set_status("加密失敗")

        threading.Thread(target=task, daemon=True).start()

    def _run_decrypt(self):
        wav = self.dec_wav.get().strip()
        keypng = self.dec_keypng.get().strip()
        pw = self.dec_pw.get().strip()
        dbv = self.dec_dbval.get().strip()
        out_dir = self.dec_out_dir.get().strip()

        errors = []

        if not os.path.exists(wav):
            errors.append("找不到 WAV 音訊")
        if not os.path.exists(keypng):
            errors.append("找不到金鑰 PNG")
        if not pw:
            errors.append("口令不能為空")
        if not os.path.isdir(out_dir):
            errors.append("請設定有效的輸出資料夾")

        try:
            db_int = int(dbv)
        except ValueError:
            errors.append("db_value 必須是整數")
            db_int = 0

        if errors:
            messagebox.showerror("輸入錯誤", "\n".join(f"• {e}" for e in errors))
            return

        self._log_clear(self.dec_log)
        self.recovered_info.configure(text="解密中...")

        def task():
            ok, result = full_decrypt(
                wav,
                keypng,
                pw,
                db_int,
                out_dir,
                progress_cb=lambda m: self._log_append(self.dec_log, m)
            )

            if ok:
                self.set_status("解密完成")
                self.recovered_info.configure(text=f"✅ 已還原檔案：\n{result}")
            else:
                self._log_append(self.dec_log, f"\n❌ 解密失敗：{result}")
                self.recovered_info.configure(text="解密失敗")
                self.set_status("解密失敗")

        threading.Thread(target=task, daemon=True).start()

    # ══════════════════════════════════════════════════════════
    #  GUI helper
    # ══════════════════════════════════════════════════════════

    def _load_txt_file(self):
        path = filedialog.askopenfilename(
            title="選擇 db_value.txt",
            filetypes=[("文字檔案", "*.txt"), ("所有檔案", "*.*")]
        )

        if not path:
            return

        try:
            db_val = load_db_value_from_txt(path)
            self.dec_txt_var.set(path)
            self.dec_dbval.delete(0, "end")
            self.dec_dbval.insert(0, str(db_val))
            self.set_status(f"✓ 已從 txt 讀入 db_value = {db_val}")

        except Exception as e:
            messagebox.showerror("讀取失敗", str(e))

    def _copy_dbval(self):
        val = self.db_value.get()

        if val:
            self.clipboard_clear()
            self.clipboard_append(val)
            self.set_status(f"已複製 db_value：{val}")

    def _section(self, parent, title, color):
        f = tk.Frame(parent, bg=C["bg"])
        f.pack(fill="x", pady=(12, 4))

        tk.Label(
            f,
            text=title,
            font=FONT_HEAD,
            fg=color,
            bg=C["bg"]
        ).pack(side="left")

        tk.Frame(
            f,
            bg=C["border"],
            height=1
        ).pack(side="left", fill="x", expand=True, padx=(8, 0), pady=6)

    def _label(self, parent, text):
        tk.Label(
            parent,
            text=text,
            font=FONT_BODY,
            fg=C["subtext"],
            bg=C["bg"]
        ).pack(anchor="w", pady=(6, 1))

    def _entry(self, parent, show=""):
        e = tk.Entry(
            parent,
            font=FONT_MONO,
            bg=C["entry_bg"],
            fg=C["text"],
            insertbackground=C["text"],
            relief="flat",
            bd=0,
            show=show
        )

        e.pack(fill="x", ipady=6, ipadx=8, pady=(0, 2))
        tk.Frame(parent, bg=C["border"], height=1).pack(fill="x")

        return e

    def _file_row(self, parent, label, btn_text, ftypes):
        self._label(parent, label)

        var = tk.StringVar()
        row = tk.Frame(parent, bg=C["bg"])
        row.pack(fill="x", pady=(0, 2))

        tk.Entry(
            row,
            textvariable=var,
            font=FONT_MONO,
            bg=C["entry_bg"],
            fg=C["text"],
            insertbackground=C["text"],
            relief="flat",
            bd=0
        ).pack(side="left", fill="x", expand=True, ipady=6, ipadx=8)

        tk.Button(
            row,
            text=btn_text,
            font=FONT_MONO,
            bg=C["border"],
            fg=C["text"],
            relief="flat",
            bd=0,
            activebackground=C["accent"],
            activeforeground="white",
            cursor="hand2",
            padx=10,
            command=lambda: var.set(filedialog.askopenfilename(filetypes=ftypes) or var.get())
        ).pack(side="right", padx=(6, 0))

        tk.Frame(parent, bg=C["border"], height=1).pack(fill="x")

        return var

    def _save_row(self, parent, label, default, ftypes):
        self._label(parent, label)

        var = tk.StringVar(value=default)
        row = tk.Frame(parent, bg=C["bg"])
        row.pack(fill="x", pady=(0, 2))

        tk.Entry(
            row,
            textvariable=var,
            font=FONT_MONO,
            bg=C["entry_bg"],
            fg=C["text"],
            insertbackground=C["text"],
            relief="flat",
            bd=0
        ).pack(side="left", fill="x", expand=True, ipady=6, ipadx=8)

        def choose_save_path():
            path = filedialog.asksaveasfilename(
                defaultextension=ftypes[0][1].lstrip("*"),
                filetypes=ftypes
            )

            if path:
                var.set(path)

        tk.Button(
            row,
            text="瀏覽",
            font=FONT_MONO,
            bg=C["border"],
            fg=C["text"],
            relief="flat",
            bd=0,
            activebackground=C["accent2"],
            activeforeground="white",
            cursor="hand2",
            padx=10,
            command=choose_save_path
        ).pack(side="right", padx=(6, 0))

        tk.Frame(parent, bg=C["border"], height=1).pack(fill="x")

        return var

    def _dir_row(self, parent, label):
        self._label(parent, label)

        var = tk.StringVar(value=os.getcwd())
        row = tk.Frame(parent, bg=C["bg"])
        row.pack(fill="x", pady=(0, 2))

        tk.Entry(
            row,
            textvariable=var,
            font=FONT_MONO,
            bg=C["entry_bg"],
            fg=C["text"],
            insertbackground=C["text"],
            relief="flat",
            bd=0
        ).pack(side="left", fill="x", expand=True, ipady=6, ipadx=8)

        tk.Button(
            row,
            text="選擇資料夾",
            font=FONT_MONO,
            bg=C["border"],
            fg=C["text"],
            relief="flat",
            bd=0,
            activebackground=C["accent4"],
            activeforeground="white",
            cursor="hand2",
            padx=10,
            command=lambda: var.set(filedialog.askdirectory() or var.get())
        ).pack(side="right", padx=(6, 0))

        tk.Frame(parent, bg=C["border"], height=1).pack(fill="x")

        return var

    def _log_box(self, parent):
        frame = tk.Frame(
            parent,
            bg=C["surface"],
            highlightbackground=C["border"],
            highlightthickness=1
        )
        frame.pack(fill="both", expand=True, pady=(4, 8))

        t = tk.Text(
            frame,
            font=FONT_MONO,
            bg=C["entry_bg"],
            fg=C["text"],
            relief="flat",
            bd=0,
            wrap="word",
            height=8,
            insertbackground=C["text"],
            state="disabled"
        )

        sb = ttk.Scrollbar(frame, command=t.yview)
        t.configure(yscrollcommand=sb.set)

        sb.pack(side="right", fill="y")
        t.pack(fill="both", expand=True, padx=1, pady=1)

        return t

    def _log_append(self, box, msg):
        try:
            box.configure(state="normal")
            box.insert("end", msg + "\n")
            box.see("end")
            box.configure(state="disabled")
            self.set_status(msg.strip())
        except tk.TclError:
            pass

    def _log_clear(self, box):
        box.configure(state="normal")
        box.delete("1.0", "end")
        box.configure(state="disabled")

    def _big_button(self, parent, text, color, cmd):
        tk.Button(
            parent,
            text=text,
            font=("Consolas", 12, "bold"),
            bg=color,
            fg="white",
            relief="flat",
            bd=0,
            activebackground=C["surface"],
            activeforeground=color,
            cursor="hand2",
            pady=10,
            command=cmd
        ).pack(fill="x", pady=(8, 4))


# ══════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════

if __name__ == "__main__":
    app = PhantasmUniversalAESApp()
    app.mainloop()
